GENERATOR CODE
--------------

In [0]:
C = ['order_id','customer_id','order_ts','quantity','unit_price','channel','status']
VOL = '/Volumes/workspace/dbsf_2570002/raw_files'
CTY = "'Chennai','Pune','Kochi','Indore','Jaipur','Bhopal','Nagpur','Surat'"; R = f'{VOL}/raw' 
def wr(df, path, cols):  
    (df.selectExpr(*cols).write.mode('overwrite').option('header', True) 
     .option('ignoreLeadingWhiteSpace', False)
     .option('ignoreTrailingWhiteSpace', False).csv(path))
def day(d, n, first, q="cast(1 + pmod(id, 5) as string)", c=None, t=None):
    return spark.range(n).selectExpr(
        f"format_string('ORD%06d', id + {first}) order_id",
        c or "format_string('C%05d', cast(pmod(id*7919, 3000) as int)) customer_id",
        t or f"cast(date_add(date'2025-06-01', {d-1}) as string) order_ts",
        f"{q} quantity",
        "round(20 + pmod(id*37, 18000)/100.0, 2) unit_price",
        "concat('CH', cast(pmod(id, 4) as string)) channel",
        "if(pmod(id, 50) = 0, 'CANCELLED', 'PLACED') status")
for d in range(1, 12):             # ten loaded files plus the clean control file
    wr(day(d, 5000, (d-1)*5000 + 1), f'{R}/orders/day_{d:02d}', C)
for d in range(1, 11):             # 300 customers a file, 3,000 in all
    wr(spark.range(300).selectExpr(
        f"format_string('C%05d', cast(id + {(d-1)*300} as int)) customer_id",
        f"element_at(array({CTY}), cast(pmod(id + {(d-1)*300}, 8) as int) + 1) city",
        f"cast(date_add(date'2025-06-01', {d-1}) as string) signup_date"),
       f'{R}/customers/cust_day_{d:02d}', ['customer_id','city','signup_date'])
wr(day(12, 5000, 55001, q="if(id < 40, 'unknown', cast(1+pmod(id,5) as string))"),
   f'{R}/orders/day_12', C)  
wr(day(13, 4850, 60001).union(day(1, 150, 1)), f'{R}/orders/day_13', C)
wr(day(14, 5000, 64851), f'{R}/orders/day_14',   
   ['order_id','order_ts','unit_price','quantity','status','channel','customer_id'])
spark.createDataFrame([(','.join(C),)], 'value string') \
    .write.mode('overwrite').text(f'{R}/orders/day_15')   
wr(day(16, 5000, 69851, c="if(id < 220, 'C99999', format_string('C%05d'," 
           " cast(pmod(id*7919, 3000) as int))) customer_id", 
           t="if(id >= 220 and id < 310, '2035-06-16', '2025-06-16') order_ts"),
   f'{R}/orders/day_16', C)

Now, we'll check the generated files

This shows the folders/files created by the generator.

In [0]:
display(dbutils.fs.ls(R))

path,name,size,modificationTime
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/,customers/,0,1790832026019
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/,orders/,0,1790832026019


Now we will check the files inside them

Customers-

In [0]:
display(dbutils.fs.ls(f"{R}/customers/"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_01/,cust_day_01/,0,1790832031724
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_02/,cust_day_02/,0,1790832031724
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_03/,cust_day_03/,0,1790832031724
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_04/,cust_day_04/,0,1790832031724
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_05/,cust_day_05/,0,1790832031724
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_06/,cust_day_06/,0,1790832031724
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_07/,cust_day_07/,0,1790832031724
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_08/,cust_day_08/,0,1790832031724
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_09/,cust_day_09/,0,1790832031724
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_10/,cust_day_10/,0,1790832031724


Orders-

In [0]:
display(dbutils.fs.ls(f"{R}/orders/"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_01/,day_01/,0,1790832033851
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_02/,day_02/,0,1790832033851
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_03/,day_03/,0,1790832033851
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_04/,day_04/,0,1790832033851
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_05/,day_05/,0,1790832033851
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_06/,day_06/,0,1790832033851
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_07/,day_07/,0,1790832033851
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_08/,day_08/,0,1790832033851
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_09/,day_09/,0,1790832033851
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_10/,day_10/,0,1790832033851


Since all the generator code is verified, we'll move on to the Bronze layer

But before that, we'll check the data of day 11

In [0]:
display(dbutils.fs.ls(f"{R}/orders/day_11/"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_11/_committed_102220687603071922,_committed_102220687603071922,1439,1790800194000
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_11/_committed_1406159239014625567,_committed_1406159239014625567,1463,1790739729000
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_11/_committed_1643348028189343596,_committed_1643348028189343596,1455,1790729911000
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_11/_committed_1994270516884993819,_committed_1994270516884993819,1455,1790733604000
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_11/_committed_2168488415125429373,_committed_2168488415125429373,1479,1790794291000
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_11/_committed_2252492844321872911,_committed_2252492844321872911,1479,1790816901000
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_11/_committed_2366158404475906526,_committed_2366158404475906526,1471,1790791301000
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_11/_committed_2726807381617252699,_committed_2726807381617252699,1479,1790795689000
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_11/_committed_323418572732148381,_committed_323418572732148381,1447,1790702244000
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_11/_committed_3633227366636265860,_committed_3633227366636265860,1455,1790737195000


Since Spark processes data in partitions, the generator code created multiple parts of CSV file instead of just one whole file

So, we'll inspect the actual data of day 11 order

In [0]:
orders_day11 = spark.read \
    .option("header", "true") \
    .schema(orders_schema) \
    .csv(f"{R}/orders/day_11")

display(orders_day11.limit(10))

order_id,customer_id,order_ts,quantity,unit_price,channel,status
ORD051251,C01750,2025-06-11,1,122.50,CH2,CANCELLED
ORD051252,C00669,2025-06-11,2,122.87,CH3,PLACED
ORD051253,C02588,2025-06-11,3,123.24,CH0,PLACED
ORD051254,C01507,2025-06-11,4,123.61,CH1,PLACED
ORD051255,C00426,2025-06-11,5,123.98,CH2,PLACED
ORD051256,C02345,2025-06-11,1,124.35,CH3,PLACED
ORD051257,C01264,2025-06-11,2,124.72,CH0,PLACED
ORD051258,C00183,2025-06-11,3,125.09,CH1,PLACED
ORD051259,C02102,2025-06-11,4,125.46,CH2,PLACED
ORD051260,C01021,2025-06-11,5,125.83,CH3,PLACED


In [0]:
print("Rows:", orders_day11.count())
print("Columns:", orders_day11.columns)

Rows: 5000
Columns: ['order_id', 'customer_id', 'order_ts', 'quantity', 'unit_price', 'channel', 'status']


The above table proves that the data from day 11 is finally cleaned and we are ready for Bronze implementation

BRONZE LAYER
------------

In order to build the bronze layer, at first we will read all the order files from all the day folders.

Now, we will save this dataframe as a Delta table, which is DataBricks' table format

In [0]:
spark.table("bronze_orders").write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("bronze_orders")

In [0]:
bronze_check = spark.table("bronze_orders")

print("Rows:", bronze_check.count())
print("Columns:", bronze_check.columns)

Rows: 75000
Columns: ['order_id', 'customer_id', 'order_ts', 'quantity', 'unit_price', 'channel', 'status', '_source_file', '_ingested_at', '_load_id', '_row_hash']


In [0]:
day14_check = (
    spark.table("bronze_orders")
    .filter(F.col("_source_file").contains("/day_14/"))
)

print("Day 14 rows:", day14_check.count())

display(day14_check.limit(10))

Day 14 rows: 5000


order_id,customer_id,order_ts,quantity,unit_price,channel,status,_source_file,_ingested_at,_load_id,_row_hash
ORD064851,C00000,2025-06-14,1,20.00,CH0,CANCELLED,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_14/part-00000-tid-7625897506670455749-03ed2cd9-fd4b-4c25-a0be-7ac9b3e02349-38456-1-c000.csv,2026-10-01T01:11:07.718Z,20261001010857796,f3bd57267415a1fef1c2a01d03e7e0188fda01ac1dfdbe64688375f46e90c56a
ORD064852,C01919,2025-06-14,2,20.37,CH1,PLACED,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_14/part-00000-tid-7625897506670455749-03ed2cd9-fd4b-4c25-a0be-7ac9b3e02349-38456-1-c000.csv,2026-10-01T01:11:07.718Z,20261001010857796,61ad6d7a0f37ba7c0eedd297cfabe15eb947302e277944ba5ff00c2b065a37df
ORD064853,C00838,2025-06-14,3,20.74,CH2,PLACED,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_14/part-00000-tid-7625897506670455749-03ed2cd9-fd4b-4c25-a0be-7ac9b3e02349-38456-1-c000.csv,2026-10-01T01:11:07.718Z,20261001010857796,0bfb44d3a8c0a0a4b5196d9be8a2aab11eb7c62134195ad6c534d20227868dc7
ORD064854,C02757,2025-06-14,4,21.11,CH3,PLACED,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_14/part-00000-tid-7625897506670455749-03ed2cd9-fd4b-4c25-a0be-7ac9b3e02349-38456-1-c000.csv,2026-10-01T01:11:07.718Z,20261001010857796,b5d3e043e6591c94284fe6575b3a992fc4d0fb0faebcee2660ea84adb63ee3ff
ORD064855,C01676,2025-06-14,5,21.48,CH0,PLACED,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_14/part-00000-tid-7625897506670455749-03ed2cd9-fd4b-4c25-a0be-7ac9b3e02349-38456-1-c000.csv,2026-10-01T01:11:07.718Z,20261001010857796,bc2849a489431d868ebd1ec80b02eccd8cce758fcb622989f4a9d8f32d4f3076
ORD064856,C00595,2025-06-14,1,21.85,CH1,PLACED,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_14/part-00000-tid-7625897506670455749-03ed2cd9-fd4b-4c25-a0be-7ac9b3e02349-38456-1-c000.csv,2026-10-01T01:11:07.718Z,20261001010857796,1d167f6020c609b1acb0a2bb0ca5a832d2b338ec1b6d86d2840836b6cad64bf7
ORD064857,C02514,2025-06-14,2,22.22,CH2,PLACED,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_14/part-00000-tid-7625897506670455749-03ed2cd9-fd4b-4c25-a0be-7ac9b3e02349-38456-1-c000.csv,2026-10-01T01:11:07.718Z,20261001010857796,0f03ca7ab0ae8a936ecb7724be25383a71459da33e957e4e5144d8b9b811f456
ORD064858,C01433,2025-06-14,3,22.59,CH3,PLACED,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_14/part-00000-tid-7625897506670455749-03ed2cd9-fd4b-4c25-a0be-7ac9b3e02349-38456-1-c000.csv,2026-10-01T01:11:07.718Z,20261001010857796,0473157cd3be1a22e66d3ad2dff9872b0602df5bae131a624689b429d1a0d481
ORD064859,C00352,2025-06-14,4,22.96,CH0,PLACED,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_14/part-00000-tid-7625897506670455749-03ed2cd9-fd4b-4c25-a0be-7ac9b3e02349-38456-1-c000.csv,2026-10-01T01:11:07.718Z,20261001010857796,ecbb80cac333ac5861863fe09c603247ee29dbb4ed4d4dd535cc0817cba9ede7
ORD064860,C02271,2025-06-14,5,23.33,CH1,PLACED,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_14/part-00000-tid-7625897506670455749-03ed2cd9-fd4b-4c25-a0be-7ac9b3e02349-38456-1-c000.csv,2026-10-01T01:11:07.718Z,20261001010857796,f8d912427d97176568a018960c381c061efa7ff0d9ee1490554793bfcfbba4f9


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType
)

orders_path = f"{R}/orders"

bronze_orders_path = f"{R}/bronze/orders"

In [0]:
orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_ts", StringType(), True),
    StructField("quantity", StringType(), True),
    StructField("unit_price", StringType(), True),
    StructField("channel", StringType(), True),
    StructField("status", StringType(), True)
])

In [0]:
load_id = spark.sql(
    "SELECT date_format(current_timestamp(), 'yyyyMMddHHmmssSSS') AS id"
).first()["id"]

print("Load ID:", load_id)

Load ID: 20261001052845951


In [0]:
day11_path = f"{orders_path}/day_11"

day11_header = (
    spark.read
        .option("header", "true")
        .csv(day11_path)
)

print(day11_header.columns)

['order_id', 'customer_id', 'order_ts', 'quantity', 'unit_price', 'channel', 'status']


In [0]:
day11_data = (
    spark.read
        .option("header", "true")
        .schema(orders_schema)
        .csv(day11_path)
)

display(day11_data.limit(5))

order_id,customer_id,order_ts,quantity,unit_price,channel,status
ORD051251,C01750,2025-06-11,1,122.50,CH2,CANCELLED
ORD051252,C00669,2025-06-11,2,122.87,CH3,PLACED
ORD051253,C02588,2025-06-11,3,123.24,CH0,PLACED
ORD051254,C01507,2025-06-11,4,123.61,CH1,PLACED
ORD051255,C00426,2025-06-11,5,123.98,CH2,PLACED


In [0]:
print(day11_header.columns)

['order_id', 'customer_id', 'order_ts', 'quantity', 'unit_price', 'channel', 'status']


In [0]:
display(day11_data.limit(5))

order_id,customer_id,order_ts,quantity,unit_price,channel,status
ORD051251,C01750,2025-06-11,1,122.50,CH2,CANCELLED
ORD051252,C00669,2025-06-11,2,122.87,CH3,PLACED
ORD051253,C02588,2025-06-11,3,123.24,CH0,PLACED
ORD051254,C01507,2025-06-11,4,123.61,CH1,PLACED
ORD051255,C00426,2025-06-11,5,123.98,CH2,PLACED


In [0]:
print(day11_data.schema)

StructType([StructField('order_id', StringType(), True), StructField('customer_id', StringType(), True), StructField('order_ts', StringType(), True), StructField('quantity', StringType(), True), StructField('unit_price', StringType(), True), StructField('channel', StringType(), True), StructField('status', StringType(), True)])


In [0]:
from pyspark.sql import functions as F

day11_bronze = (
    day11_data
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_load_id", F.lit(load_id))
    .withColumn(
        "_row_hash",
        F.sha2(
            F.concat_ws(
                "||",
                *[F.col(c) for c in day11_data.columns]
            ),
            256
        )
    )
)

display(day11_bronze.limit(5))

order_id,customer_id,order_ts,quantity,unit_price,channel,status,_source_file,_ingested_at,_load_id,_row_hash
ORD051251,C01750,2025-06-11,1,122.50,CH2,CANCELLED,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_11/part-00002-tid-8258936191124440434-d9c4634b-9187-403e-9206-def7044d1aee-214-1-c000.csv,2026-10-01T05:30:22.694Z,20261001052845951,472d734f9a935c62224ee0e1109d94e93e5f658c0fa763d2b8c3edd7e27373e0
ORD051252,C00669,2025-06-11,2,122.87,CH3,PLACED,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_11/part-00002-tid-8258936191124440434-d9c4634b-9187-403e-9206-def7044d1aee-214-1-c000.csv,2026-10-01T05:30:22.694Z,20261001052845951,d8cf626a28d23a4073cd49a8ee14a367d0c16b032e5fe46aff94bc64b3d8a05a
ORD051253,C02588,2025-06-11,3,123.24,CH0,PLACED,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_11/part-00002-tid-8258936191124440434-d9c4634b-9187-403e-9206-def7044d1aee-214-1-c000.csv,2026-10-01T05:30:22.694Z,20261001052845951,3b1ed1264151287d0e74cf01749cfd7834fd2a09658d05f9d3f94600f7a6e0b6
ORD051254,C01507,2025-06-11,4,123.61,CH1,PLACED,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_11/part-00002-tid-8258936191124440434-d9c4634b-9187-403e-9206-def7044d1aee-214-1-c000.csv,2026-10-01T05:30:22.694Z,20261001052845951,f0725cce38337c8d591bffc70c2311acc0759a10721a637e43313662fab8f186
ORD051255,C00426,2025-06-11,5,123.98,CH2,PLACED,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_11/part-00002-tid-8258936191124440434-d9c4634b-9187-403e-9206-def7044d1aee-214-1-c000.csv,2026-10-01T05:30:22.694Z,20261001052845951,745ba1571e15d44181884ff14262ab913382546c758f090665eae61f02eeaf34


In [0]:
from pyspark.sql import functions as F

day11_bronze = (
    day11_data
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_load_id", F.lit(load_id))
    .withColumn(
        "_row_hash",
        F.sha2(
            F.concat_ws(
                "||",
                *[F.col(c) for c in day11_data.columns]
            ),
            256
        )
    )
)

display(day11_bronze.limit(5))

order_id,customer_id,order_ts,quantity,unit_price,channel,status,_source_file,_ingested_at,_load_id,_row_hash
ORD051251,C01750,2025-06-11,1,122.50,CH2,CANCELLED,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_11/part-00002-tid-8258936191124440434-d9c4634b-9187-403e-9206-def7044d1aee-214-1-c000.csv,2026-10-01T05:30:30.000Z,20261001052845951,472d734f9a935c62224ee0e1109d94e93e5f658c0fa763d2b8c3edd7e27373e0
ORD051252,C00669,2025-06-11,2,122.87,CH3,PLACED,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_11/part-00002-tid-8258936191124440434-d9c4634b-9187-403e-9206-def7044d1aee-214-1-c000.csv,2026-10-01T05:30:30.000Z,20261001052845951,d8cf626a28d23a4073cd49a8ee14a367d0c16b032e5fe46aff94bc64b3d8a05a
ORD051253,C02588,2025-06-11,3,123.24,CH0,PLACED,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_11/part-00002-tid-8258936191124440434-d9c4634b-9187-403e-9206-def7044d1aee-214-1-c000.csv,2026-10-01T05:30:30.000Z,20261001052845951,3b1ed1264151287d0e74cf01749cfd7834fd2a09658d05f9d3f94600f7a6e0b6
ORD051254,C01507,2025-06-11,4,123.61,CH1,PLACED,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_11/part-00002-tid-8258936191124440434-d9c4634b-9187-403e-9206-def7044d1aee-214-1-c000.csv,2026-10-01T05:30:30.000Z,20261001052845951,f0725cce38337c8d591bffc70c2311acc0759a10721a637e43313662fab8f186
ORD051255,C00426,2025-06-11,5,123.98,CH2,PLACED,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_11/part-00002-tid-8258936191124440434-d9c4634b-9187-403e-9206-def7044d1aee-214-1-c000.csv,2026-10-01T05:30:30.000Z,20261001052845951,745ba1571e15d44181884ff14262ab913382546c758f090665eae61f02eeaf34


In [0]:
display(
    day11_bronze.select(
        "_source_file",
        "_ingested_at",
        "_load_id",
        "_row_hash"
    ).limit(5)
)

_source_file,_ingested_at,_load_id,_row_hash
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_11/part-00002-tid-8258936191124440434-d9c4634b-9187-403e-9206-def7044d1aee-214-1-c000.csv,2026-10-01T05:31:03.731Z,20261001052845951,472d734f9a935c62224ee0e1109d94e93e5f658c0fa763d2b8c3edd7e27373e0
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_11/part-00002-tid-8258936191124440434-d9c4634b-9187-403e-9206-def7044d1aee-214-1-c000.csv,2026-10-01T05:31:03.731Z,20261001052845951,d8cf626a28d23a4073cd49a8ee14a367d0c16b032e5fe46aff94bc64b3d8a05a
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_11/part-00002-tid-8258936191124440434-d9c4634b-9187-403e-9206-def7044d1aee-214-1-c000.csv,2026-10-01T05:31:03.731Z,20261001052845951,3b1ed1264151287d0e74cf01749cfd7834fd2a09658d05f9d3f94600f7a6e0b6
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_11/part-00002-tid-8258936191124440434-d9c4634b-9187-403e-9206-def7044d1aee-214-1-c000.csv,2026-10-01T05:31:03.731Z,20261001052845951,f0725cce38337c8d591bffc70c2311acc0759a10721a637e43313662fab8f186
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_11/part-00002-tid-8258936191124440434-d9c4634b-9187-403e-9206-def7044d1aee-214-1-c000.csv,2026-10-01T05:31:03.731Z,20261001052845951,745ba1571e15d44181884ff14262ab913382546c758f090665eae61f02eeaf34


In [0]:
print(day11_bronze.columns)

['order_id', 'customer_id', 'order_ts', 'quantity', 'unit_price', 'channel', 'status', '_source_file', '_ingested_at', '_load_id', '_row_hash']


In [0]:
display(
    day11_bronze.select(
        "_ingested_at",
        "_load_id",
        "_row_hash"
    ).limit(5)
)

_ingested_at,_load_id,_row_hash
2026-10-01T05:31:28.664Z,20261001052845951,472d734f9a935c62224ee0e1109d94e93e5f658c0fa763d2b8c3edd7e27373e0
2026-10-01T05:31:28.664Z,20261001052845951,d8cf626a28d23a4073cd49a8ee14a367d0c16b032e5fe46aff94bc64b3d8a05a
2026-10-01T05:31:28.664Z,20261001052845951,3b1ed1264151287d0e74cf01749cfd7834fd2a09658d05f9d3f94600f7a6e0b6
2026-10-01T05:31:28.664Z,20261001052845951,f0725cce38337c8d591bffc70c2311acc0759a10721a637e43313662fab8f186
2026-10-01T05:31:28.664Z,20261001052845951,745ba1571e15d44181884ff14262ab913382546c758f090665eae61f02eeaf34


Process all 16 days data

Building the load manifest - for that at first we need to create a small list of the 16 logical order loads.

In [0]:
order_days = [f"day_{i:02d}" for i in range(1, 17)]

print(order_days)

['day_01', 'day_02', 'day_03', 'day_04', 'day_05', 'day_06', 'day_07', 'day_08', 'day_09', 'day_10', 'day_11', 'day_12', 'day_13', 'day_14', 'day_15', 'day_16']


Then, we will create the manifest records

In [0]:
manifest_rows = []

for day in order_days:
    day_path = f"{orders_path}/{day}"
    
    # Read header only, without schema
    df_header = (
        spark.read
            .option("header", "true")
            .csv(day_path)
    )
    
    # Number of data rows
    rows = df_header.count()
    
    # Give each order day its own load_id
    day_load_id = f"{load_id}_{day}"
    
    manifest_rows.append((
        day_load_id,
        "orders",
        day_path,
        rows,
        ",".join(df_header.columns),
        None
    ))

from pyspark.sql.types import LongType

orders_manifest = spark.createDataFrame(
    manifest_rows,
    StructType([
        StructField("load_id", StringType(), True),
        StructField("table_name", StringType(), True),
        StructField("source_file", StringType(), True),
        StructField("rows_in_file", LongType(), True),
        StructField("actual_columns", StringType(), True),
        StructField("notes", StringType(), True),
    ])
)

display(orders_manifest)

load_id,table_name,source_file,rows_in_file,actual_columns,notes
20261001052845951_day_01,orders,/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_01,5000,"order_id,customer_id,order_ts,quantity,unit_price,channel,status",null
20261001052845951_day_02,orders,/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_02,5000,"order_id,customer_id,order_ts,quantity,unit_price,channel,status",null
20261001052845951_day_03,orders,/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_03,5000,"order_id,customer_id,order_ts,quantity,unit_price,channel,status",null
20261001052845951_day_04,orders,/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_04,5000,"order_id,customer_id,order_ts,quantity,unit_price,channel,status",null
20261001052845951_day_05,orders,/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_05,5000,"order_id,customer_id,order_ts,quantity,unit_price,channel,status",null
20261001052845951_day_06,orders,/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_06,5000,"order_id,customer_id,order_ts,quantity,unit_price,channel,status",null
20261001052845951_day_07,orders,/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_07,5000,"order_id,customer_id,order_ts,quantity,unit_price,channel,status",null
20261001052845951_day_08,orders,/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_08,5000,"order_id,customer_id,order_ts,quantity,unit_price,channel,status",null
20261001052845951_day_09,orders,/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_09,5000,"order_id,customer_id,order_ts,quantity,unit_price,channel,status",null
20261001052845951_day_10,orders,/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_10,5000,"order_id,customer_id,order_ts,quantity,unit_price,channel,status",null


Inspect the important values before saving the manifest

In [0]:
for row in manifest_rows:
    print(row)

('20261001052845951_day_01', 'orders', '/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_01', 5000, 'order_id,customer_id,order_ts,quantity,unit_price,channel,status', None)
('20261001052845951_day_02', 'orders', '/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_02', 5000, 'order_id,customer_id,order_ts,quantity,unit_price,channel,status', None)
('20261001052845951_day_03', 'orders', '/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_03', 5000, 'order_id,customer_id,order_ts,quantity,unit_price,channel,status', None)
('20261001052845951_day_04', 'orders', '/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_04', 5000, 'order_id,customer_id,order_ts,quantity,unit_price,channel,status', None)
('20261001052845951_day_05', 'orders', '/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_05', 5000, 'order_id,customer_id,order_ts,quantity,unit_price,channel,status', None)
('20261001052845951_day_06', 'orders', '/Volumes/workspace/dbsf_2570002/raw_files/raw/orde

Fix the manifest timestamp

In [0]:
from pyspark.sql.types import LongType, TimestampType

manifest_df = spark.createDataFrame(
    manifest_rows,
    StructType([
        StructField("load_id", StringType(), True),
        StructField("table_name", StringType(), True),
        StructField("source_file", StringType(), True),
        StructField("rows_in_file", LongType(), True),
        StructField("column_list", StringType(), True),
        StructField("ingested_at", TimestampType(), True),
    ])
).withColumn(
    "ingested_at",
    F.current_timestamp()
)

display(manifest_df)

load_id,table_name,source_file,rows_in_file,column_list,ingested_at
20261001052845951_day_01,orders,/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_01,5000,"order_id,customer_id,order_ts,quantity,unit_price,channel,status",2026-10-01T05:32:39.382Z
20261001052845951_day_02,orders,/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_02,5000,"order_id,customer_id,order_ts,quantity,unit_price,channel,status",2026-10-01T05:32:39.382Z
20261001052845951_day_03,orders,/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_03,5000,"order_id,customer_id,order_ts,quantity,unit_price,channel,status",2026-10-01T05:32:39.382Z
20261001052845951_day_04,orders,/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_04,5000,"order_id,customer_id,order_ts,quantity,unit_price,channel,status",2026-10-01T05:32:39.382Z
20261001052845951_day_05,orders,/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_05,5000,"order_id,customer_id,order_ts,quantity,unit_price,channel,status",2026-10-01T05:32:39.382Z
20261001052845951_day_06,orders,/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_06,5000,"order_id,customer_id,order_ts,quantity,unit_price,channel,status",2026-10-01T05:32:39.382Z
20261001052845951_day_07,orders,/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_07,5000,"order_id,customer_id,order_ts,quantity,unit_price,channel,status",2026-10-01T05:32:39.382Z
20261001052845951_day_08,orders,/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_08,5000,"order_id,customer_id,order_ts,quantity,unit_price,channel,status",2026-10-01T05:32:39.382Z
20261001052845951_day_09,orders,/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_09,5000,"order_id,customer_id,order_ts,quantity,unit_price,channel,status",2026-10-01T05:32:39.382Z
20261001052845951_day_10,orders,/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_10,5000,"order_id,customer_id,order_ts,quantity,unit_price,channel,status",2026-10-01T05:32:39.382Z


In [0]:
display(
    manifest_df
    .select("load_id", "source_file", "rows_in_file")
    .orderBy("load_id")
)

load_id,source_file,rows_in_file
20261001052845951_day_01,/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_01,5000
20261001052845951_day_02,/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_02,5000
20261001052845951_day_03,/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_03,5000
20261001052845951_day_04,/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_04,5000
20261001052845951_day_05,/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_05,5000
20261001052845951_day_06,/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_06,5000
20261001052845951_day_07,/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_07,5000
20261001052845951_day_08,/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_08,5000
20261001052845951_day_09,/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_09,5000
20261001052845951_day_10,/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_10,5000


In [0]:
customer_manifest_check = spark.table("bronze_load_manifest")

display(
    customer_manifest_check
    .filter("table_name = 'customers'")
    .select("load_id", "source_file", "rows_in_file")
    .orderBy("source_file")
)

load_id,source_file,rows_in_file
20261001052845951_cust_day_01,/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_01,300
20261001052845951_cust_day_02,/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_02,300
20261001052845951_cust_day_03,/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_03,300
20261001052845951_cust_day_04,/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_04,300
20261001052845951_cust_day_05,/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_05,300
20261001052845951_cust_day_06,/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_06,300
20261001052845951_cust_day_07,/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_07,300
20261001052845951_cust_day_08,/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_08,300
20261001052845951_cust_day_09,/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_09,300
20261001052845951_cust_day_10,/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_10,300


Save the manifest

In [0]:
manifest_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_load_manifest")

In [0]:
manifest_check = spark.table("bronze_load_manifest")

print("Manifest rows:", manifest_check.count())

display(
    manifest_check.orderBy("source_file")
)

Manifest rows: 16


load_id,table_name,source_file,rows_in_file,column_list,ingested_at
20261001052845951_day_01,orders,/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_01,5000,"order_id,customer_id,order_ts,quantity,unit_price,channel,status",2026-10-01T05:33:39.742Z
20261001052845951_day_02,orders,/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_02,5000,"order_id,customer_id,order_ts,quantity,unit_price,channel,status",2026-10-01T05:33:39.742Z
20261001052845951_day_03,orders,/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_03,5000,"order_id,customer_id,order_ts,quantity,unit_price,channel,status",2026-10-01T05:33:39.742Z
20261001052845951_day_04,orders,/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_04,5000,"order_id,customer_id,order_ts,quantity,unit_price,channel,status",2026-10-01T05:33:39.742Z
20261001052845951_day_05,orders,/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_05,5000,"order_id,customer_id,order_ts,quantity,unit_price,channel,status",2026-10-01T05:33:39.742Z
20261001052845951_day_06,orders,/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_06,5000,"order_id,customer_id,order_ts,quantity,unit_price,channel,status",2026-10-01T05:33:39.742Z
20261001052845951_day_07,orders,/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_07,5000,"order_id,customer_id,order_ts,quantity,unit_price,channel,status",2026-10-01T05:33:39.742Z
20261001052845951_day_08,orders,/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_08,5000,"order_id,customer_id,order_ts,quantity,unit_price,channel,status",2026-10-01T05:33:39.742Z
20261001052845951_day_09,orders,/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_09,5000,"order_id,customer_id,order_ts,quantity,unit_price,channel,status",2026-10-01T05:33:39.742Z
20261001052845951_day_10,orders,/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_10,5000,"order_id,customer_id,order_ts,quantity,unit_price,channel,status",2026-10-01T05:33:39.742Z


Save all_orders as bronze_orders

In [0]:
bronze_orders_check = spark.table("bronze_orders")

print("Rows:", bronze_orders_check.count())
print("Columns:", bronze_orders_check.columns)

display(
    bronze_orders_check
    .filter(F.col("_source_file").contains("/day_14/"))
    .select(
        "order_id",
        "customer_id",
        "order_ts",
        "quantity",
        "unit_price",
        "channel",
        "status"
    )
    .limit(10)
)

Rows: 75000
Columns: ['order_id', 'customer_id', 'order_ts', 'quantity', 'unit_price', 'channel', 'status', '_source_file', '_ingested_at', '_load_id', '_row_hash']


order_id,customer_id,order_ts,quantity,unit_price,channel,status
ORD064851,C00000,2025-06-14,1,20.00,CH0,CANCELLED
ORD064852,C01919,2025-06-14,2,20.37,CH1,PLACED
ORD064853,C00838,2025-06-14,3,20.74,CH2,PLACED
ORD064854,C02757,2025-06-14,4,21.11,CH3,PLACED
ORD064855,C01676,2025-06-14,5,21.48,CH0,PLACED
ORD064856,C00595,2025-06-14,1,21.85,CH1,PLACED
ORD064857,C02514,2025-06-14,2,22.22,CH2,PLACED
ORD064858,C01433,2025-06-14,3,22.59,CH3,PLACED
ORD064859,C00352,2025-06-14,4,22.96,CH0,PLACED
ORD064860,C02271,2025-06-14,5,23.33,CH1,PLACED


Bronze_orders is completed. Hence, we will start the same process with bronze_customers

Define the customer schema - 

In [0]:
customer_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("city", StringType(), True),
    StructField("signup_date", StringType(), True)
])

customers_path = f"{R}/customers"

print(customer_schema)

StructType([StructField('customer_id', StringType(), True), StructField('city', StringType(), True), StructField('signup_date', StringType(), True)])


Test the header read - 

In [0]:
cust_day1_path = f"{customers_path}/cust_day_01"

cust_day1_header = (
    spark.read
        .option("header", "true")
        .csv(cust_day1_path)
)

print(cust_day1_header.columns)

['customer_id', 'city', 'signup_date']


Read customer data with the schema - 

In [0]:
cust_day1_data = (
    spark.read
        .option("header", "true")
        .schema(customer_schema)
        .csv(cust_day1_path)
)

display(cust_day1_data.limit(5))

customer_id,city,signup_date
C00187,Indore,2025-06-01
C00188,Jaipur,2025-06-01
C00189,Bhopal,2025-06-01
C00190,Nagpur,2025-06-01
C00191,Surat,2025-06-01


In [0]:
print(cust_day1_data.schema)

StructType([StructField('customer_id', StringType(), True), StructField('city', StringType(), True), StructField('signup_date', StringType(), True)])


Add the required Bronze metadata

In [0]:
cust_day1_bronze = (
    cust_day1_data
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_load_id", F.lit(load_id))
    .withColumn(
        "_row_hash",
        F.sha2(
            F.concat_ws(
                "||",
                *[F.col(c) for c in customer_schema.fieldNames()]
            ),
            256
        )
    )
)

print(cust_day1_bronze.columns)

['customer_id', 'city', 'signup_date', '_source_file', '_ingested_at', '_load_id', '_row_hash']


In [0]:
display(
    cust_day1_bronze.select(
        "_source_file",
        "_ingested_at",
        "_load_id",
        "_row_hash"
    ).limit(5)
)

_source_file,_ingested_at,_load_id,_row_hash
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_01/part-00005-tid-2748562983010415908-5fd728a6-aa4c-444a-939c-0e342f711f73-220-1-c000.csv,2026-10-01T05:35:01.954Z,20261001052845951,a9f61c0be6bacef6cec6c832e0c9214e778a8488cec40b7319d619e91e8f89e7
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_01/part-00005-tid-2748562983010415908-5fd728a6-aa4c-444a-939c-0e342f711f73-220-1-c000.csv,2026-10-01T05:35:01.954Z,20261001052845951,9eac1f2dab7f494707c2e075f79b434d3f51bd9ad168d330862fe6250c72740a
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_01/part-00005-tid-2748562983010415908-5fd728a6-aa4c-444a-939c-0e342f711f73-220-1-c000.csv,2026-10-01T05:35:01.954Z,20261001052845951,d8942059ad8c4bbba8cbaefe716ee1f438e0d6f35496fa8621868f9aa4661609
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_01/part-00005-tid-2748562983010415908-5fd728a6-aa4c-444a-939c-0e342f711f73-220-1-c000.csv,2026-10-01T05:35:01.954Z,20261001052845951,57fe5dc9e0bbace144c23e6fe8dd063478d39046061b4f2a12c3c6a8a6db5534
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_01/part-00005-tid-2748562983010415908-5fd728a6-aa4c-444a-939c-0e342f711f73-220-1-c000.csv,2026-10-01T05:35:01.954Z,20261001052845951,245c3f20d0dc5526ca712ea8f12235ecb0bdcfe2844b36bbe399a346f64df6e8


Now, we will read all customer files using the declared schema

Now, we will save the proper Bronze Customers table

In [0]:
customer_check = spark.table("bronze_customers")

print("Rows:", customer_check.count())
print("Columns:", customer_check.columns)

Rows: 3000
Columns: ['customer_id', 'city', 'signup_date', '_source_file', '_ingested_at', '_load_id', '_row_hash']


As per the above tables, we are finished with the original bronze data tables. Now, we'll add the 10 customer loads to bronze_load_manifest

In [0]:
customer_days = [f"cust_day_{i:02d}" for i in range(1, 11)]
customer_manifest_rows = []

for day in customer_days:
    day_path = f"{customers_path}/{day}"

    df_header = (
        spark.read
        .option("header", "true")
        .csv(day_path)
    )

    rows = df_header.count()

    day_load_id = f"{load_id}_{day}"

    customer_manifest_rows.append((
        day_load_id,
        "customers",
        day_path,
        rows,
        ",".join(df_header.columns),
        None
    ))

from pyspark.sql.types import LongType

customers_manifest = spark.createDataFrame(
    customer_manifest_rows,
    StructType([
        StructField("load_id", StringType(), True),
        StructField("table_name", StringType(), True),
        StructField("source_file", StringType(), True),
        StructField("rows_in_file", LongType(), True),
        StructField("actual_columns", StringType(), True),
        StructField("notes", StringType(), True),
    ])
)

display(customers_manifest)

load_id,table_name,source_file,rows_in_file,actual_columns,notes
20261001052845951_cust_day_01,customers,/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_01,300,"customer_id,city,signup_date",null
20261001052845951_cust_day_02,customers,/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_02,300,"customer_id,city,signup_date",null
20261001052845951_cust_day_03,customers,/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_03,300,"customer_id,city,signup_date",null
20261001052845951_cust_day_04,customers,/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_04,300,"customer_id,city,signup_date",null
20261001052845951_cust_day_05,customers,/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_05,300,"customer_id,city,signup_date",null
20261001052845951_cust_day_06,customers,/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_06,300,"customer_id,city,signup_date",null
20261001052845951_cust_day_07,customers,/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_07,300,"customer_id,city,signup_date",null
20261001052845951_cust_day_08,customers,/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_08,300,"customer_id,city,signup_date",null
20261001052845951_cust_day_09,customers,/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_09,300,"customer_id,city,signup_date",null
20261001052845951_cust_day_10,customers,/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_10,300,"customer_id,city,signup_date",null


Then we'll combine with the existing orders manifest

In [0]:
all_manifest_rows = manifest_rows + customer_manifest_rows

print("Total manifest records:", len(all_manifest_rows))

Total manifest records: 26


Now we'll create the final manifest 

In [0]:
from pyspark.sql.types import LongType, TimestampType

final_manifest = (
    spark.createDataFrame(
        all_manifest_rows,
        StructType([
            StructField("load_id", StringType(), True),
            StructField("table_name", StringType(), True),
            StructField("source_file", StringType(), True),
            StructField("rows_in_file", LongType(), True),
            StructField("column_list", StringType(), True),
            StructField("ingested_at", TimestampType(), True),
        ])
    )
    .withColumn("ingested_at", F.current_timestamp())
)

display(final_manifest.orderBy("table_name", "source_file"))

load_id,table_name,source_file,rows_in_file,column_list,ingested_at
20261001052845951_cust_day_01,customers,/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_01,300,"customer_id,city,signup_date",2026-10-01T05:36:07.448Z
20261001052845951_cust_day_02,customers,/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_02,300,"customer_id,city,signup_date",2026-10-01T05:36:07.448Z
20261001052845951_cust_day_03,customers,/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_03,300,"customer_id,city,signup_date",2026-10-01T05:36:07.448Z
20261001052845951_cust_day_04,customers,/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_04,300,"customer_id,city,signup_date",2026-10-01T05:36:07.448Z
20261001052845951_cust_day_05,customers,/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_05,300,"customer_id,city,signup_date",2026-10-01T05:36:07.448Z
20261001052845951_cust_day_06,customers,/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_06,300,"customer_id,city,signup_date",2026-10-01T05:36:07.448Z
20261001052845951_cust_day_07,customers,/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_07,300,"customer_id,city,signup_date",2026-10-01T05:36:07.448Z
20261001052845951_cust_day_08,customers,/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_08,300,"customer_id,city,signup_date",2026-10-01T05:36:07.448Z
20261001052845951_cust_day_09,customers,/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_09,300,"customer_id,city,signup_date",2026-10-01T05:36:07.448Z
20261001052845951_cust_day_10,customers,/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_10,300,"customer_id,city,signup_date",2026-10-01T05:36:07.448Z


Now, we will replace the old manifest with the one as the old one had orders-only manifest

In [0]:
final_manifest.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_load_manifest")

In [0]:
manifest_check = spark.table("bronze_load_manifest")

print("Total manifest rows:", manifest_check.count())

display(
    manifest_check.groupBy("table_name")
                  .count()
                  .orderBy("table_name")
)

Total manifest rows: 26


table_name,count
customers,10
orders,16


Since the customers have 10 and orders have 16 count, we can say Bronze layer is done and we can move on to Silver layer where the data quality will be checked.

But as per the instructions, "Read by declared schema, never inferSchema and never by position". Our table currently reads the CSV columns positionally which could be a problem as Day 14 has columns in the wrong order

To fix that, we will rebuild Bronze table using column names correctly

In [0]:
orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_ts", StringType(), True),
    StructField("quantity", StringType(), True),
    StructField("unit_price", StringType(), True),
    StructField("channel", StringType(), True),
    StructField("status", StringType(), True)
])

In [0]:
expected_order_cols = [
    "order_id",
    "customer_id",
    "order_ts",
    "quantity",
    "unit_price",
    "channel",
    "status"
]

all_orders_by_name = None

for day in [f"day_{i:02d}" for i in range(1, 17)]:

    day_path = f"{orders_path}/{day}"

    # Read header first to know the actual column names/order
    header_df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .csv(day_path)
    )

    actual_cols = header_df.columns

    print(f"{day} columns:", actual_cols)

    # Read data WITHOUT applying schema positionally
    data_df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .csv(day_path)
    )

    # Select using COLUMN NAMES
    day_df = data_df.select(
        *[
            F.col(c).cast("string").alias(c)
            for c in expected_order_cols
        ],
        F.col("_metadata.file_path").alias("_source_file")
    )

    if all_orders_by_name is None:
        all_orders_by_name = day_df
    else:
        all_orders_by_name = all_orders_by_name.unionByName(day_df)


print(
    "Bronze orders rows:",
    all_orders_by_name.count()
)

print(
    "Bronze orders columns:",
    all_orders_by_name.columns
)

display(
    all_orders_by_name.limit(10)
)

day_01 columns: ['order_id', 'customer_id', 'order_ts', 'quantity', 'unit_price', 'channel', 'status']
day_02 columns: ['order_id', 'customer_id', 'order_ts', 'quantity', 'unit_price', 'channel', 'status']
day_03 columns: ['order_id', 'customer_id', 'order_ts', 'quantity', 'unit_price', 'channel', 'status']
day_04 columns: ['order_id', 'customer_id', 'order_ts', 'quantity', 'unit_price', 'channel', 'status']
day_05 columns: ['order_id', 'customer_id', 'order_ts', 'quantity', 'unit_price', 'channel', 'status']
day_06 columns: ['order_id', 'customer_id', 'order_ts', 'quantity', 'unit_price', 'channel', 'status']
day_07 columns: ['order_id', 'customer_id', 'order_ts', 'quantity', 'unit_price', 'channel', 'status']
day_08 columns: ['order_id', 'customer_id', 'order_ts', 'quantity', 'unit_price', 'channel', 'status']
day_09 columns: ['order_id', 'customer_id', 'order_ts', 'quantity', 'unit_price', 'channel', 'status']
day_10 columns: ['order_id', 'customer_id', 'order_ts', 'quantity', 'unit

order_id,customer_id,order_ts,quantity,unit_price,channel,status,_source_file
ORD001251,C01750,2025-06-01,1,122.50,CH2,CANCELLED,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_01/part-00002-tid-6463461151714208560-7ff03fb9-80ad-494d-ae74-d027237750ab-134-1-c000.csv
ORD001252,C00669,2025-06-01,2,122.87,CH3,PLACED,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_01/part-00002-tid-6463461151714208560-7ff03fb9-80ad-494d-ae74-d027237750ab-134-1-c000.csv
ORD001253,C02588,2025-06-01,3,123.24,CH0,PLACED,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_01/part-00002-tid-6463461151714208560-7ff03fb9-80ad-494d-ae74-d027237750ab-134-1-c000.csv
ORD001254,C01507,2025-06-01,4,123.61,CH1,PLACED,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_01/part-00002-tid-6463461151714208560-7ff03fb9-80ad-494d-ae74-d027237750ab-134-1-c000.csv
ORD001255,C00426,2025-06-01,5,123.98,CH2,PLACED,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_01/part-00002-tid-6463461151714208560-7ff03fb9-80ad-494d-ae74-d027237750ab-134-1-c000.csv
ORD001256,C02345,2025-06-01,1,124.35,CH3,PLACED,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_01/part-00002-tid-6463461151714208560-7ff03fb9-80ad-494d-ae74-d027237750ab-134-1-c000.csv
ORD001257,C01264,2025-06-01,2,124.72,CH0,PLACED,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_01/part-00002-tid-6463461151714208560-7ff03fb9-80ad-494d-ae74-d027237750ab-134-1-c000.csv
ORD001258,C00183,2025-06-01,3,125.09,CH1,PLACED,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_01/part-00002-tid-6463461151714208560-7ff03fb9-80ad-494d-ae74-d027237750ab-134-1-c000.csv
ORD001259,C02102,2025-06-01,4,125.46,CH2,PLACED,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_01/part-00002-tid-6463461151714208560-7ff03fb9-80ad-494d-ae74-d027237750ab-134-1-c000.csv
ORD001260,C01021,2025-06-01,5,125.83,CH3,PLACED,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_01/part-00002-tid-6463461151714208560-7ff03fb9-80ad-494d-ae74-d027237750ab-134-1-c000.csv


In [0]:
all_orders_by_name = (
    all_orders_by_name
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_load_id", F.lit(load_id))
    .withColumn(
        "_row_hash",
        F.sha2(
            F.concat_ws(
                "||",
                *[F.col(c) for c in expected_order_cols]
            ),
            256
        )
    )
)

In [0]:
print("Rows:", all_orders_by_name.count())
print("Columns:", all_orders_by_name.columns)

Rows: 75000
Columns: ['order_id', 'customer_id', 'order_ts', 'quantity', 'unit_price', 'channel', 'status', '_source_file', '_ingested_at', '_load_id', '_row_hash']


In [0]:
all_orders_by_name.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("bronze_orders")

SILVER LAYER
------------

In order to start with Silver layer, at first we'll apply the Silver Rule R1, which means if an order has no order_id, we will reject that row.

We will start with creating the R1 check

In [0]:
from pyspark.sql import functions as F

silver_orders = spark.table("bronze_orders")

r1_rejected = silver_orders.filter(
    F.col("order_id").isNull() |
    (F.trim(F.col("order_id")) == "")
)

print("R1 rejected rows:", r1_rejected.count())

R1 rejected rows: 0


Since the no. of rejected rows is 0, we'll check the valid rows

In [0]:
silver_orders = silver_orders.filter(
    F.col("order_id").isNotNull() &
    (F.trim(F.col("order_id")) != "")
)

print("Rows remaining:", silver_orders.count())

Rows remaining: 75000


Since we have cleaned the null order_id, now we will check for duplicate order_id

In [0]:
duplicate_ids = (
    silver_orders
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate order IDs:", duplicate_ids.count())

Duplicate order IDs: 150


In [0]:
r2_rejected = (
    silver_orders
    .join(
        duplicate_ids.select("order_id"),
        on="order_id",
        how="inner"
    )
)

print("R2 rejected rows:", r2_rejected.count())

R2 rejected rows: 300


We got 300 rejected rows which is factually wrong, and it came due to double counting of two rows each. 
Meaning -
150 duplicate IDs × 2 rows each = 300 rows

To fix that, we will reject the copied rows instead of original rows. We'll first get the original rows, find the copied 150 rows from day 13 and then reject them only

In [0]:
original_orders = silver_orders.filter(
    ~F.col("_source_file").contains("/day_13/")
)

print("Original orders:", original_orders.count())

Original orders: 70000


In [0]:
previous_order_ids = (
    silver_orders
    .filter(
        F.col("_source_file").rlike("/day_(0[1-9]|1[0-2])/")
    )
    .select("order_id")
    .distinct()
)

print("Previous unique order IDs:", previous_order_ids.count())

Previous unique order IDs: 60000


In [0]:
r2_rejected = (
    silver_orders
    .filter(F.col("_source_file").contains("/day_13/"))
    .join(
        previous_order_ids,
        on="order_id",
        how="inner"
    )
)

print("R2 rejected rows:", r2_rejected.count())

R2 rejected rows: 150


In [0]:
silver_orders = (
    silver_orders
    .join(
        r2_rejected.select("_source_file", "order_id"),
        on=["_source_file", "order_id"],
        how="left_anti"
    )
)

print("Rows remaining:", silver_orders.count())

Rows remaining: 74850


Now that all the duplicate order_ids are removed, we'll begin with R3, which is to cast the quantity to numeric value

In [0]:
silver_orders = silver_orders.withColumn(
    "quantity_int",
    F.col("quantity").try_cast("int")
)

r3_rejected = silver_orders.filter(
    F.col("quantity").isNotNull() &
    F.col("quantity_int").isNull()
)

print("R3 rejected rows:", r3_rejected.count())

R3 rejected rows: 40


In [0]:
display(
    silver_orders
    .filter(F.col("_source_file").contains("/day_12/"))
    .select("order_id", "quantity", "_source_file")
    .limit(10)
)

order_id,quantity,_source_file
ORD055001,unknown,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_12/part-00000-tid-1684934211078485833-db53c068-e0e6-4981-9c29-869aa4e35077-301-1-c000.csv
ORD055002,unknown,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_12/part-00000-tid-1684934211078485833-db53c068-e0e6-4981-9c29-869aa4e35077-301-1-c000.csv
ORD055003,unknown,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_12/part-00000-tid-1684934211078485833-db53c068-e0e6-4981-9c29-869aa4e35077-301-1-c000.csv
ORD055004,unknown,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_12/part-00000-tid-1684934211078485833-db53c068-e0e6-4981-9c29-869aa4e35077-301-1-c000.csv
ORD055005,unknown,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_12/part-00000-tid-1684934211078485833-db53c068-e0e6-4981-9c29-869aa4e35077-301-1-c000.csv
ORD055006,unknown,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_12/part-00000-tid-1684934211078485833-db53c068-e0e6-4981-9c29-869aa4e35077-301-1-c000.csv
ORD055007,unknown,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_12/part-00000-tid-1684934211078485833-db53c068-e0e6-4981-9c29-869aa4e35077-301-1-c000.csv
ORD055008,unknown,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_12/part-00000-tid-1684934211078485833-db53c068-e0e6-4981-9c29-869aa4e35077-301-1-c000.csv
ORD055009,unknown,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_12/part-00000-tid-1684934211078485833-db53c068-e0e6-4981-9c29-869aa4e35077-301-1-c000.csv
ORD055010,unknown,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_12/part-00000-tid-1684934211078485833-db53c068-e0e6-4981-9c29-869aa4e35077-301-1-c000.csv


In [0]:
display(
    silver_orders
    .filter(F.col("_source_file").contains("/day_12/"))
    .select("order_id", "quantity", "quantity_int")
    .limit(10)
)

order_id,quantity,quantity_int
ORD055001,unknown,null
ORD055002,unknown,null
ORD055003,unknown,null
ORD055004,unknown,null
ORD055005,unknown,null
ORD055006,unknown,null
ORD055007,unknown,null
ORD055008,unknown,null
ORD055009,unknown,null
ORD055010,unknown,null


In [0]:
print(
    "Non-numeric quantities:",
    silver_orders.filter(
        F.col("quantity").isNotNull() &
        F.col("quantity_int").isNull()
    ).count()
)

Non-numeric quantities: 40


In [0]:
silver_orders = silver_orders.filter(
    F.col("quantity_int").isNotNull()
)

print("Rows remaining:", silver_orders.count())

Rows remaining: 74810


In [0]:
silver_orders = (
    silver_orders
    .drop("quantity")
    .withColumnRenamed("quantity_int", "quantity")
)

In [0]:
print(silver_orders.schema)

StructType([StructField('_source_file', StringType(), True), StructField('order_id', StringType(), True), StructField('customer_id', StringType(), True), StructField('order_ts', StringType(), True), StructField('unit_price', StringType(), True), StructField('channel', StringType(), True), StructField('status', StringType(), True), StructField('_ingested_at', TimestampType(), True), StructField('_load_id', StringType(), True), StructField('_row_hash', StringType(), True), StructField('quantity', IntegerType(), True)])


So, R3 is completed now as quantity is casted as IntegerType. Now, we have to remove a customer_id = 'C99999', which exists in 220 rows but does not exist in the customer master. 

In order to do that, we will compare silver layer's customer_id with the bronze layer's customer_id and if an order's customer doesn't exist in the customer master, we'll reject it.

In [0]:
customer_ids = (
    spark.table("bronze_customers")
    .select("customer_id")
    .distinct()
)

print("Customer IDs:", customer_ids.count())

Customer IDs: 3000


In [0]:
r4_rejected = (
    silver_orders
    .join(
        customer_ids,
        on="customer_id",
        how="left_anti"
    )
)

print("R4 rejected rows:", r4_rejected.count())

R4 rejected rows: 220


In [0]:
display(
    r4_rejected
    .groupBy("customer_id")
    .count()
)

customer_id,count
C99999,220


Total rejected rows up until now has been = 150+40+220 = 410.
Now we'll calculate the remaining rows after the rejected rows

In [0]:
silver_orders = (
    silver_orders
    .join(
        r4_rejected.select("order_id"),
        on="order_id",
        how="left_anti"
    )
)

print("Rows remaining:", silver_orders.count())

Rows remaining: 74590


Since we are done with R4, we'll move on with R5 where we check for invalid order_ts

But before that we will convert order_ts from string to date

In [0]:
silver_orders = silver_orders.withColumn(
    "order_date",
    F.to_date(F.col("order_ts"), "yyyy-MM-dd")
)

In [0]:
display(
    silver_orders
    .select("order_ts", "order_date")
    .limit(10)
)

order_ts,order_date
2025-06-05,2025-06-05
2025-06-05,2025-06-05
2025-06-05,2025-06-05
2025-06-05,2025-06-05
2025-06-05,2025-06-05
2025-06-05,2025-06-05
2025-06-05,2025-06-05
2025-06-05,2025-06-05
2025-06-05,2025-06-05
2025-06-05,2025-06-05


So order_date is now a proper DateType. As per the instructions, "a separate 90 rows carry order_ts 2035-06-16, rejected as 'order_ts_out_of_range'. Both come from day_16 and the two sets never overlap. Range-check 2024-01-01 to current_date rather than hunting for the literal year 2035."

At first, find the 90 invalid rows

In [0]:
silver_orders = silver_orders.withColumn(
    "order_date",
    F.try_to_date(F.col("order_ts"), "yyyy-MM-dd")
)

r5_rejected = silver_orders.filter(
    (F.col("order_date") < F.lit("2024-01-01").cast("date")) |
    (F.col("order_date") > F.current_date())
)

print("R5 rejected rows:", r5_rejected.count())

R5 rejected rows: 90


In [0]:
display(
    r5_rejected
    .groupBy("order_date")
    .count()
)

order_date,count
2035-06-16,90


Now we'll remove these 90 rows

In [0]:
silver_orders = silver_orders.join(
    r5_rejected.select("order_id"),
    on="order_id",
    how="left_anti"
)

print("Rows remaining:", silver_orders.count())

Rows remaining: 74500


In [0]:
from pyspark.sql import functions as F

silver_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_orders")

silver_check = spark.table("silver_orders")

print("Silver rows:", silver_check.count())

assert silver_check.count() == 74500

print("PASS: silver_orders contains exactly 74,500 rows.")

Silver rows: 74500
PASS: silver_orders contains exactly 74,500 rows.


R5 is complete and Silver layer has exact 74500 rows

In [0]:
# Verify R1-R5 rejection counts

r1_count = r1_rejected.count()
r2_count = r2_rejected.count()
r3_count = r3_rejected.count()
r4_count = r4_rejected.count()
r5_count = r5_rejected.count()

print("R1:", r1_count)
print("R2:", r2_count)
print("R3:", r3_count)
print("R4:", r4_count)
print("R5:", r5_count)

total_rejected = (
    r1_count +
    r2_count +
    r3_count +
    r4_count +
    r5_count
)

print("Total R1-R5 rejects:", total_rejected)

assert r1_count == 0
assert r2_count == 150
assert r3_count == 40
assert r4_count == 220
assert r5_count == 90
assert total_rejected == 500

print("PASS: R1-R5 totals are exactly 500.")

R1: 0
R2: 150
R3: 40
R4: 220
R5: 90
Total R1-R5 rejects: 500
PASS: R1-R5 totals are exactly 500.


Now, we'll move on with R6 where we will check File-level schema/column-order

In [0]:
expected_columns = "order_id,customer_id,order_ts,quantity,unit_price,channel,status"

In [0]:
r6_rejected = spark.table("bronze_load_manifest").filter(
    (F.col("table_name") == "orders") &
    (F.col("column_list") != expected_columns)
)

display(r6_rejected)

load_id,table_name,source_file,rows_in_file,column_list,ingested_at
20261001052845951_day_14,orders,/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_14,5000,"order_id,order_ts,unit_price,quantity,status,channel,customer_id",2026-10-01T05:36:36.388Z


In [0]:
# Validate R6

r6_count = r6_rejected.count()

print("R6 rejected files:", r6_count)

assert r6_count == 1

print("PASS: R6 rejected exactly 1 file.")

R6 rejected files: 1
PASS: R6 rejected exactly 1 file.


In [0]:

r6_day14_count = (
    r6_rejected
    .filter(F.col("source_file").contains("/day_14"))
    .count()
)

assert r6_day14_count == 1

print("PASS: R6 correctly identified day_14.")

PASS: R6 correctly identified day_14.


In [0]:
orders_manifest = (
    spark.table("bronze_load_manifest")
    .filter(F.col("table_name") == "orders")
)

display(
    orders_manifest
    .select("source_file", "rows_in_file")
    .orderBy("source_file")
)

source_file,rows_in_file
/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_01,5000
/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_02,5000
/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_03,5000
/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_04,5000
/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_05,5000
/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_06,5000
/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_07,5000
/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_08,5000
/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_09,5000
/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_10,5000


Now, as per the instructions, R7 should compare each file's rows_in_file against the median of the previous 7 loads, with a ±40% band.

In [0]:
# R7: Compare each file against the median of the previous 7 loads

from pyspark.sql.window import Window
from pyspark.sql import functions as F

orders_manifest = (
    spark.table("bronze_load_manifest")
    .filter(F.col("table_name") == "orders")
)

# Extract day number so the loads can be ordered
orders_manifest = orders_manifest.withColumn(
    "day_number",
    F.regexp_extract(
        F.col("source_file"),
        r"/day_(\d+)$",
        1
    ).cast("int")
)

# Window containing the previous 7 loads
previous_7_window = (
    Window
    .orderBy("day_number")
    .rowsBetween(-7, -1)
)

# Calculate previous-7 median
orders_manifest = orders_manifest.withColumn(
    "previous_7_median",
    F.expr(
        "percentile_approx(rows_in_file, 0.5)"
    ).over(previous_7_window)
)

# ±40% allowed range
orders_manifest = orders_manifest.withColumn(
    "lower_bound",
    F.col("previous_7_median") * 0.60
).withColumn(
    "upper_bound",
    F.col("previous_7_median") * 1.40
)

# R7 fails when the current file is outside that range
r7_rejected = orders_manifest.filter(
    F.col("previous_7_median").isNotNull() &
    (
        (F.col("rows_in_file") < F.col("lower_bound")) |
        (F.col("rows_in_file") > F.col("upper_bound"))
    )
)

display(
    orders_manifest.select(
        "source_file",
        "rows_in_file",
        "previous_7_median",
        "lower_bound",
        "upper_bound"
    ).orderBy("day_number")
)

print("R7 rejected files:", r7_rejected.count())

display(
    r7_rejected.select(
        "source_file",
        "rows_in_file",
        "previous_7_median",
        "lower_bound",
        "upper_bound"
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


source_file,rows_in_file,previous_7_median,lower_bound,upper_bound
/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_01,5000,null,null,null
/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_02,5000,5000,3000.0,7000.0
/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_03,5000,5000,3000.0,7000.0
/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_04,5000,5000,3000.0,7000.0
/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_05,5000,5000,3000.0,7000.0
/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_06,5000,5000,3000.0,7000.0
/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_07,5000,5000,3000.0,7000.0
/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_08,5000,5000,3000.0,7000.0
/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_09,5000,5000,3000.0,7000.0
/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_10,5000,5000,3000.0,7000.0


R7 rejected files: 1


source_file,rows_in_file,previous_7_median,lower_bound,upper_bound
/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_15,0,5000,3000.0,7000.0


In [0]:
# Verify the six new order loads: day_11 to day_16

new_six = (
    orders_manifest
    .filter(
        F.col("day_number").between(11, 16)
    )
)

display(
    new_six
    .select(
        "source_file",
        "rows_in_file"
    )
    .orderBy("day_number")
)

# Assert that all six files exist
assert new_six.count() == 6

# Assert the row counts
expected_counts = [5000, 5000, 5000, 5000, 0, 5000]

actual_counts = [
    row["rows_in_file"]
    for row in new_six
        .orderBy("day_number")
        .select("rows_in_file")
        .collect()
]

assert actual_counts == expected_counts

# Assert that day_15 exists and contains 0 rows
day15_rows = (
    new_six
    .filter(F.col("day_number") == 15)
    .select("rows_in_file")
    .first()["rows_in_file"]
)

assert day15_rows == 0

# Assert total = 25,000
total_new_rows = (
    new_six
    .agg(F.sum("rows_in_file").alias("total"))
    .first()["total"]
)

assert total_new_rows == 25000

print("PASS: Six new files exist.")
print("PASS: day_15 exists and contains 0 rows.")
print("PASS: Six new files contain exactly 25,000 rows.")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


source_file,rows_in_file
/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_11,5000
/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_12,5000
/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13,5000
/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_14,5000
/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_15,0
/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_16,5000


PASS: Six new files exist.
PASS: day_15 exists and contains 0 rows.
PASS: Six new files contain exactly 25,000 rows.


In [0]:
# Verify Bronze -> Silver subtraction

bronze_count = spark.table("bronze_orders").count()
silver_count = silver_orders.count()
reject_count = spark.table("silver_order_rejections").count()

assert reject_count == 500
assert bronze_count - reject_count == silver_count
assert silver_count == 74500

print("Bronze rows:", bronze_count)
print("Silver rejects:", reject_count)
print("Silver rows:", silver_count)
print(f"PASS: {bronze_count} - {reject_count} = {silver_count}")

Bronze rows: 75000
Silver rejects: 500
Silver rows: 74500
PASS: 75000 - 500 = 74500


In [0]:

print("R7 rejected files:", r7_rejected.count())

display(
    r7_rejected.select(
        "source_file",
        "rows_in_file",
        "previous_7_median",
        "lower_bound",
        "upper_bound"
    ).orderBy("day_number")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


R7 rejected files: 1


source_file,rows_in_file,previous_7_median,lower_bound,upper_bound
/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_15,0,5000,3000.0,7000.0


In [0]:
# Validate R7

r7_count = r7_rejected.count()

assert r7_count == 1

assert (
    r7_rejected
    .filter(F.col("source_file").contains("/day_15"))
    .count()
    == 1
)

print("PASS: R7 rejected exactly 1 file.")
print("PASS: R7 correctly identified day_15.")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


PASS: R7 rejected exactly 1 file.
PASS: R7 correctly identified day_15.


In [0]:
# MAGIC %md
# MAGIC Create the file-level rejection table for R6 and R7

In [0]:
file_rejections = (
    r6_rejected
    .select(
        F.col("load_id"),
        F.col("table_name"),
        F.col("source_file"),
        F.col("rows_in_file"),
        F.col("column_list")
    )
    .withColumn("rule_id", F.lit("R6"))
    .withColumn("reason", F.lit("wrong_column_contract"))
)

file_rejections = file_rejections.unionByName(
    r7_rejected
    .select(
        F.col("load_id"),
        F.col("table_name"),
        F.col("source_file"),
        F.col("rows_in_file"),
        F.col("column_list")
    )
    .withColumn("rule_id", F.lit("R7"))
    .withColumn("reason", F.lit("unexpected_file_row_count"))
)

display(
    file_rejections
    .select(
        "rule_id",
        "reason",
        "source_file",
        "rows_in_file"
    )
    .orderBy("rule_id")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


rule_id,reason,source_file,rows_in_file
R6,wrong_column_contract,/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_14,5000
R7,unexpected_file_row_count,/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_15,0


In [0]:
file_rejections.write \
    .mode("overwrite") \
    .saveAsTable("silver_file_rejections")

print(
    "File-level rejection records:",
    spark.table("silver_file_rejections").count()
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


File-level rejection records: 2


In [0]:
file_rejection_check = spark.table("silver_file_rejections")

print(
    "Total file-level rejections:",
    file_rejection_check.count()
)

assert file_rejection_check.count() == 2

Total file-level rejections: 2


In [0]:
display(
    r2_rejected.select(
        "_load_id",
        "order_id",
        "_source_file"
    ).limit(10)
)

_load_id,order_id,_source_file
20261001052845951,ORD000001,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13/part-00008-tid-5207715487933315393-663b8328-67dc-4282-abeb-f68fab272c0b-305-1-c000.csv
20261001052845951,ORD000002,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13/part-00008-tid-5207715487933315393-663b8328-67dc-4282-abeb-f68fab272c0b-305-1-c000.csv
20261001052845951,ORD000003,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13/part-00008-tid-5207715487933315393-663b8328-67dc-4282-abeb-f68fab272c0b-305-1-c000.csv
20261001052845951,ORD000004,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13/part-00008-tid-5207715487933315393-663b8328-67dc-4282-abeb-f68fab272c0b-305-1-c000.csv
20261001052845951,ORD000005,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13/part-00008-tid-5207715487933315393-663b8328-67dc-4282-abeb-f68fab272c0b-305-1-c000.csv
20261001052845951,ORD000006,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13/part-00008-tid-5207715487933315393-663b8328-67dc-4282-abeb-f68fab272c0b-305-1-c000.csv
20261001052845951,ORD000007,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13/part-00008-tid-5207715487933315393-663b8328-67dc-4282-abeb-f68fab272c0b-305-1-c000.csv
20261001052845951,ORD000008,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13/part-00008-tid-5207715487933315393-663b8328-67dc-4282-abeb-f68fab272c0b-305-1-c000.csv
20261001052845951,ORD000009,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13/part-00008-tid-5207715487933315393-663b8328-67dc-4282-abeb-f68fab272c0b-305-1-c000.csv
20261001052845951,ORD000010,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13/part-00008-tid-5207715487933315393-663b8328-67dc-4282-abeb-f68fab272c0b-305-1-c000.csv


Now, let's build rejection tables

In [0]:
r2_rejections = r2_rejected.select(
    F.col("_load_id").alias("load_id"),
    F.col("order_id"),
    F.col("_source_file").alias("source_file")
).withColumn(
    "rule_id", F.lit("R2")
).withColumn(
    "reason", F.lit("duplicate_order_id")
)

In [0]:
display(r2_rejections.limit(10))

load_id,order_id,source_file,rule_id,reason
20261001052845951,ORD000001,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13/part-00008-tid-5207715487933315393-663b8328-67dc-4282-abeb-f68fab272c0b-305-1-c000.csv,R2,duplicate_order_id
20261001052845951,ORD000002,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13/part-00008-tid-5207715487933315393-663b8328-67dc-4282-abeb-f68fab272c0b-305-1-c000.csv,R2,duplicate_order_id
20261001052845951,ORD000003,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13/part-00008-tid-5207715487933315393-663b8328-67dc-4282-abeb-f68fab272c0b-305-1-c000.csv,R2,duplicate_order_id
20261001052845951,ORD000004,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13/part-00008-tid-5207715487933315393-663b8328-67dc-4282-abeb-f68fab272c0b-305-1-c000.csv,R2,duplicate_order_id
20261001052845951,ORD000005,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13/part-00008-tid-5207715487933315393-663b8328-67dc-4282-abeb-f68fab272c0b-305-1-c000.csv,R2,duplicate_order_id
20261001052845951,ORD000006,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13/part-00008-tid-5207715487933315393-663b8328-67dc-4282-abeb-f68fab272c0b-305-1-c000.csv,R2,duplicate_order_id
20261001052845951,ORD000007,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13/part-00008-tid-5207715487933315393-663b8328-67dc-4282-abeb-f68fab272c0b-305-1-c000.csv,R2,duplicate_order_id
20261001052845951,ORD000008,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13/part-00008-tid-5207715487933315393-663b8328-67dc-4282-abeb-f68fab272c0b-305-1-c000.csv,R2,duplicate_order_id
20261001052845951,ORD000009,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13/part-00008-tid-5207715487933315393-663b8328-67dc-4282-abeb-f68fab272c0b-305-1-c000.csv,R2,duplicate_order_id
20261001052845951,ORD000010,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13/part-00008-tid-5207715487933315393-663b8328-67dc-4282-abeb-f68fab272c0b-305-1-c000.csv,R2,duplicate_order_id


In [0]:
print("R2 rejection records:", r2_rejections.count())

R2 rejection records: 150


Next, let’s build the complete row-level rejection/quarantine table for R1–R5.

In [0]:
from pyspark.sql import functions as F

original_columns = [
    "order_id",
    "customer_id",
    "order_ts",
    "quantity",
    "unit_price",
    "channel",
    "status",
    "_source_file",
    "_ingested_at",
    "_load_id",
    "_row_hash"
]

r1_rejections = r1_rejected.select(original_columns).withColumn(
    "rule_id", F.lit("R1")
).withColumn(
    "reason", F.lit("null_order_id")
)

r2_rejections = r2_rejected.select(original_columns).withColumn(
    "rule_id", F.lit("R2")
).withColumn(
    "reason", F.lit("duplicate_order_id")
)

r3_rejections = r3_rejected.select(original_columns).withColumn(
    "rule_id", F.lit("R3")
).withColumn(
    "reason", F.lit("quantity_not_numeric")
)

r4_rejections = r4_rejected.select(original_columns).withColumn(
    "quantity", F.col("quantity").cast("string")
).withColumn(
    "rule_id", F.lit("R4")
).withColumn(
    "reason", F.lit("unknown_customer")
)

r5_rejections = r5_rejected.select(original_columns).withColumn(
    "quantity", F.col("quantity").cast("string")
).withColumn(
    "rule_id", F.lit("R5")
).withColumn(
    "reason", F.lit("order_ts_out_of_range")
)

In [0]:
rejection_table = (
    r1_rejections
    .unionByName(r2_rejections)
    .unionByName(r3_rejections)
    .unionByName(r4_rejections)
    .unionByName(r5_rejections)
)

print("Total rejected rows:", rejection_table.count())

display(
    rejection_table
    .groupBy("rule_id", "reason")
    .count()
    .orderBy("rule_id")
)

Total rejected rows: 500


rule_id,reason,count
R2,duplicate_order_id,150
R3,quantity_not_numeric,40
R4,unknown_customer,220
R5,order_ts_out_of_range,90


In [0]:
rejection_table.write.mode("overwrite").saveAsTable("silver_order_rejections")

In [0]:
# Validate file-level rejection table

file_rejection_check = spark.table("silver_file_rejections")

print(
    "Total file-level rejections:",
    file_rejection_check.count()
)

assert file_rejection_check.count() == 2

assert (
    file_rejection_check
    .filter(F.col("rule_id") == "R6")
    .count()
    == 1
)

assert (
    file_rejection_check
    .filter(F.col("rule_id") == "R7")
    .count()
    == 1
)

print("PASS: silver_file_rejections contains exactly 2 files.")
print("PASS: R6 = 1 file and R7 = 1 file.")

Total file-level rejections: 2
PASS: silver_file_rejections contains exactly 2 files.
PASS: R6 = 1 file and R7 = 1 file.


In [0]:
display(
    file_rejection_check
    .select(
        "rule_id",
        "reason",
        "source_file",
        "rows_in_file"
    )
    .orderBy("rule_id")
)

rule_id,reason,source_file,rows_in_file
R6,wrong_column_contract,/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_14,5000
R7,unexpected_file_row_count,/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_15,0


In [0]:

overlapping_rejects = (
    spark.table("silver_order_rejections")
    .groupBy("_row_hash")
    .agg(
        F.countDistinct("rule_id").alias("rule_count")
    )
    .filter(F.col("rule_count") > 1)
)

overlap_count = overlapping_rejects.count()

print("Rows rejected by multiple rules:", overlap_count)

assert overlap_count == 0

print("PASS: No row is counted twice across R1-R5.")

Rows rejected by multiple rules: 0
PASS: No row is counted twice across R1-R5.


In [0]:
print("Rejection table rows:", spark.table("silver_order_rejections").count())

Rejection table rows: 500


In [0]:
file_rejections.write.mode("overwrite").saveAsTable(
    "silver_file_rejections"
)

print(
    "File-level rejection records:",
    spark.table("silver_file_rejections").count()
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


File-level rejection records: 2


Now, we'll create the file-level rejection table which will identify problematic files.

In [0]:
file_rejections = (
    r6_rejected
    .select(
        F.col("load_id"),
        F.col("table_name"),
        F.col("source_file"),
        F.col("rows_in_file"),
        F.col("column_list")
    )
    .withColumn("rule_id", F.lit("R6"))
    .withColumn("reason", F.lit("wrong_column_contract"))
)

file_rejections = file_rejections.unionByName(
    r7_rejected
    .select(
        F.col("load_id"),
        F.col("table_name"),
        F.col("source_file"),
        F.col("rows_in_file"),
        F.col("column_list")
    )
    .withColumn("rule_id", F.lit("R7"))
    .withColumn("reason", F.lit("unexpected_file_row_count"))
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
display(
    r7_rejected.select(
        "source_file",
        "rows_in_file"
    )
)

# COMMAND ----------

# COMMAND ----------

display(
    r2_rejected.select(
        "_load_id",
        "order_id",
        "_source_file"
    ).limit(10)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


source_file,rows_in_file
/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_15,0


_load_id,order_id,_source_file
20261001052845951,ORD000001,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13/part-00008-tid-5207715487933315393-663b8328-67dc-4282-abeb-f68fab272c0b-305-1-c000.csv
20261001052845951,ORD000002,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13/part-00008-tid-5207715487933315393-663b8328-67dc-4282-abeb-f68fab272c0b-305-1-c000.csv
20261001052845951,ORD000003,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13/part-00008-tid-5207715487933315393-663b8328-67dc-4282-abeb-f68fab272c0b-305-1-c000.csv
20261001052845951,ORD000004,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13/part-00008-tid-5207715487933315393-663b8328-67dc-4282-abeb-f68fab272c0b-305-1-c000.csv
20261001052845951,ORD000005,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13/part-00008-tid-5207715487933315393-663b8328-67dc-4282-abeb-f68fab272c0b-305-1-c000.csv
20261001052845951,ORD000006,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13/part-00008-tid-5207715487933315393-663b8328-67dc-4282-abeb-f68fab272c0b-305-1-c000.csv
20261001052845951,ORD000007,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13/part-00008-tid-5207715487933315393-663b8328-67dc-4282-abeb-f68fab272c0b-305-1-c000.csv
20261001052845951,ORD000008,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13/part-00008-tid-5207715487933315393-663b8328-67dc-4282-abeb-f68fab272c0b-305-1-c000.csv
20261001052845951,ORD000009,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13/part-00008-tid-5207715487933315393-663b8328-67dc-4282-abeb-f68fab272c0b-305-1-c000.csv
20261001052845951,ORD000010,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13/part-00008-tid-5207715487933315393-663b8328-67dc-4282-abeb-f68fab272c0b-305-1-c000.csv


In [0]:
display(
    r7_rejected.select(
        "source_file",
        "rows_in_file"
    )
)

# COMMAND ----------

display(
    r2_rejected.select(
        "_load_id",
        "order_id",
        "_source_file"
    ).limit(10)
)

# COMMAND ----------

# MAGIC %md
# MAGIC Now, let's build rejection tables

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


source_file,rows_in_file
/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_15,0


_load_id,order_id,_source_file
20261001052845951,ORD000001,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13/part-00008-tid-5207715487933315393-663b8328-67dc-4282-abeb-f68fab272c0b-305-1-c000.csv
20261001052845951,ORD000002,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13/part-00008-tid-5207715487933315393-663b8328-67dc-4282-abeb-f68fab272c0b-305-1-c000.csv
20261001052845951,ORD000003,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13/part-00008-tid-5207715487933315393-663b8328-67dc-4282-abeb-f68fab272c0b-305-1-c000.csv
20261001052845951,ORD000004,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13/part-00008-tid-5207715487933315393-663b8328-67dc-4282-abeb-f68fab272c0b-305-1-c000.csv
20261001052845951,ORD000005,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13/part-00008-tid-5207715487933315393-663b8328-67dc-4282-abeb-f68fab272c0b-305-1-c000.csv
20261001052845951,ORD000006,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13/part-00008-tid-5207715487933315393-663b8328-67dc-4282-abeb-f68fab272c0b-305-1-c000.csv
20261001052845951,ORD000007,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13/part-00008-tid-5207715487933315393-663b8328-67dc-4282-abeb-f68fab272c0b-305-1-c000.csv
20261001052845951,ORD000008,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13/part-00008-tid-5207715487933315393-663b8328-67dc-4282-abeb-f68fab272c0b-305-1-c000.csv
20261001052845951,ORD000009,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13/part-00008-tid-5207715487933315393-663b8328-67dc-4282-abeb-f68fab272c0b-305-1-c000.csv
20261001052845951,ORD000010,dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13/part-00008-tid-5207715487933315393-663b8328-67dc-4282-abeb-f68fab272c0b-305-1-c000.csv


In [0]:
display(
    file_rejections
    .select(
        "rule_id",
        "reason",
        "source_file",
        "rows_in_file"
    )
    .orderBy("rule_id")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


rule_id,reason,source_file,rows_in_file
R6,wrong_column_contract,/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_14,5000
R7,unexpected_file_row_count,/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_15,0


Now we'll save the rejection tables

In [0]:
file_rejections.write.mode("overwrite").saveAsTable(
    "silver_file_rejections"
)

print(
    "File-level rejection records:",
    spark.table("silver_file_rejections").count()
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


File-level rejection records: 2


In [0]:
print(
    "File-level rejection records:",
    spark.table("silver_file_rejections").count()
)

File-level rejection records: 2


Now we will create DQ level metrics, which will give our dashboard the main numbers it needs.

In [0]:
from pyspark.sql import functions as F

dq_rule_metrics = (
    rejection_table
    .groupBy("rule_id", "reason")
    .agg(
        F.count("*").alias("rejected_rows")
    )
    .orderBy("rule_id")
)

display(dq_rule_metrics)

rule_id,reason,rejected_rows
R2,duplicate_order_id,150
R3,quantity_not_numeric,40
R4,unknown_customer,220
R5,order_ts_out_of_range,90


In [0]:
dq_rule_metrics.write.mode("overwrite").saveAsTable(
    "dq_rule_metrics"
)

Then we'll create the DQ summary

In [0]:
total_bronze = spark.table("bronze_orders").count()
total_silver = silver_orders.count()
total_rejected = spark.table("silver_order_rejections").count()

dq_summary = spark.createDataFrame(
    [
        ("total_bronze_rows", total_bronze),
        ("valid_silver_rows", total_silver),
        ("rejected_rows", total_rejected)
    ],
    ["metric", "value"]
)

display(dq_summary)

metric,value
total_bronze_rows,75000
valid_silver_rows,74500
rejected_rows,500


In [0]:
dq_summary.write.mode("overwrite").saveAsTable(
    "dq_summary"
)

In [0]:
print("Silver orders:", silver_orders.count())
print("Silver rejects:", spark.table("silver_order_rejections").count())

Silver orders: 74500
Silver rejects: 500


In [0]:
print("Bronze orders:", spark.table("bronze_orders").count())
print("Silver orders:", silver_orders.count())
print("Silver rejects:", spark.table("silver_order_rejections").count())

Bronze orders: 75000
Silver orders: 74500
Silver rejects: 500


Our core silver layer and data quality layer is now built, where we got -

1. silver_orders - 74,500 valid orders
2. silver_order_rejections - 500 rejected rows
3. silver_file_rejections - 2 rejected files
4. dq_rule_metrics - rejection count by rule
5. dq_summary - overall Bronze/Silver/rejection counts

Now we will move on with the Gold Layer where we will make business table metrics that will eventually feed our pipeline health & data quality dashboard.

GOLD LAYER
----------

As per the instructions, there are two tables specified -
1. gold_dq_results
2. gold_freshness

First, we'll inspect the customer manifest before creating the gold_dq_results

In [0]:
customers_manifest = (
    spark.table("bronze_load_manifest")
    .filter(F.col("table_name") == "customers")
)

display(
    customers_manifest
    .select(
        "load_id",
        "source_file",
        "rows_in_file",
        "ingested_at"
    )
    .orderBy("source_file")
)

load_id,source_file,rows_in_file,ingested_at
20261001052845951_cust_day_01,/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_01,300,2026-10-01T05:36:36.388Z
20261001052845951_cust_day_02,/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_02,300,2026-10-01T05:36:36.388Z
20261001052845951_cust_day_03,/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_03,300,2026-10-01T05:36:36.388Z
20261001052845951_cust_day_04,/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_04,300,2026-10-01T05:36:36.388Z
20261001052845951_cust_day_05,/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_05,300,2026-10-01T05:36:36.388Z
20261001052845951_cust_day_06,/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_06,300,2026-10-01T05:36:36.388Z
20261001052845951_cust_day_07,/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_07,300,2026-10-01T05:36:36.388Z
20261001052845951_cust_day_08,/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_08,300,2026-10-01T05:36:36.388Z
20261001052845951_cust_day_09,/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_09,300,2026-10-01T05:36:36.388Z
20261001052845951_cust_day_10,/Volumes/workspace/dbsf_2570002/raw_files/raw/customers/cust_day_10,300,2026-10-01T05:36:36.388Z


Now, create the rule definitions

In [0]:
order_rules = [
    ("R1", "null_key", "row"),
    ("R2", "duplicate_order_id", "row"),
    ("R3", "quantity_not_numeric", "row"),
    ("R4", "unknown_customer", "row"),
    ("R5", "order_ts_out_of_range", "row"),
    ("R6", "column_contract", "file"),
    ("R7", "row_count_vs_median", "file")
]

customer_rules = [
    ("R1", "null_key", "row"),
    ("R7", "row_count_vs_median", "file")
]

print("Order rules:", len(order_rules))
print("Customer rules:", len(customer_rules))

Order rules: 7
Customer rules: 2


Now, we'll create the Orders portion of gold_dq_results

In [0]:
orders_manifest = (
    spark.table("bronze_load_manifest")
    .filter(F.col("table_name") == "orders")
)

order_loads = orders_manifest.select(
    "load_id",
    "source_file",
    "rows_in_file"
).distinct()

print("Order loads:", order_loads.count())

Order loads: 16


Now let's find out exactly which files contain the R2–R5 failures

In [0]:
print("R2 failures by file")
display(
    r2_rejected
    .groupBy("_source_file")
    .count()
    .orderBy("_source_file")
)

print("R3 failures by file")
display(
    r3_rejected
    .groupBy("_source_file")
    .count()
    .orderBy("_source_file")
)

print("R4 failures by file")
display(
    r4_rejected
    .groupBy("_source_file")
    .count()
    .orderBy("_source_file")
)

print("R5 failures by file")
display(
    r5_rejected
    .groupBy("_source_file")
    .count()
    .orderBy("_source_file")
)

R2 failures by file


_source_file,count
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13/part-00008-tid-5207715487933315393-663b8328-67dc-4282-abeb-f68fab272c0b-305-1-c000.csv,18
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13/part-00009-tid-5207715487933315393-663b8328-67dc-4282-abeb-f68fab272c0b-317-1-c000.csv,19
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13/part-00010-tid-5207715487933315393-663b8328-67dc-4282-abeb-f68fab272c0b-306-1-c000.csv,19
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13/part-00011-tid-5207715487933315393-663b8328-67dc-4282-abeb-f68fab272c0b-318-1-c000.csv,19
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13/part-00012-tid-5207715487933315393-663b8328-67dc-4282-abeb-f68fab272c0b-307-1-c000.csv,18
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13/part-00013-tid-5207715487933315393-663b8328-67dc-4282-abeb-f68fab272c0b-319-1-c000.csv,19
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13/part-00014-tid-5207715487933315393-663b8328-67dc-4282-abeb-f68fab272c0b-308-1-c000.csv,19
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_13/part-00015-tid-5207715487933315393-663b8328-67dc-4282-abeb-f68fab272c0b-320-1-c000.csv,19


R3 failures by file


_source_file,count
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_12/part-00000-tid-1684934211078485833-db53c068-e0e6-4981-9c29-869aa4e35077-301-1-c000.csv,40


R4 failures by file


_source_file,count
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_16/part-00000-tid-3661126087540093203-bfa9bca1-7a9f-493c-ba2a-6685a26569c3-330-1-c000.csv,220


R5 failures by file


_source_file,count
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/raw/orders/day_16/part-00000-tid-3661126087540093203-bfa9bca1-7a9f-493c-ba2a-6685a26569c3-330-1-c000.csv,90


Now let's build Orders Gold results 

In [0]:
from pyspark.sql import functions as F

# ============================================================
# GOLD ORDERS — 16 LOADS × 7 RULES
# ============================================================

# ------------------------------------------------------------
# 1. All order loads from manifest
# ------------------------------------------------------------
order_loads = (
    orders_manifest
    .select(
        "load_id",
        "source_file",
        "rows_in_file"
    )
    .distinct()
)

# Manifest paths are already directory paths.
# Remove dbfs: if present.
order_loads = order_loads.withColumn(
    "source_file",
    F.regexp_replace(
        F.col("source_file"),
        "^dbfs:",
        ""
    )
)

# ------------------------------------------------------------
# 2. Rule definitions
# ------------------------------------------------------------
order_rules_df = spark.createDataFrame(
    order_rules,
    ["rule_id", "rule_name", "rule_scope"]
)

# ------------------------------------------------------------
# 3. Create 16 × 7 = 112 rows
# ------------------------------------------------------------
gold_orders = order_loads.crossJoin(order_rules_df)

# ------------------------------------------------------------
# 4. Function to convert rejected-row file path
#    into the manifest directory path
# ------------------------------------------------------------
def normalize_rejected_path(col_name):
    return F.regexp_replace(
        F.regexp_replace(
            F.col(col_name),
            "^dbfs:",
            ""
        ),
        r"/part-.*\.csv$",
        ""
    )

# ------------------------------------------------------------
# 5. R1
# ------------------------------------------------------------
r1_failures = (
    r1_rejected
    .select(
        normalize_rejected_path("_source_file").alias("source_file"),
        F.lit("R1").alias("rule_id")
    )
    .groupBy("source_file", "rule_id")
    .count()
    .withColumnRenamed("count", "rows_failed")
)

# ------------------------------------------------------------
# 6. R2
# ------------------------------------------------------------
r2_failures = (
    r2_rejected
    .select(
        normalize_rejected_path("_source_file").alias("source_file"),
        F.lit("R2").alias("rule_id")
    )
    .groupBy("source_file", "rule_id")
    .count()
    .withColumnRenamed("count", "rows_failed")
)

# ------------------------------------------------------------
# 7. R3
# ------------------------------------------------------------
r3_failures = (
    r3_rejected
    .select(
        normalize_rejected_path("_source_file").alias("source_file"),
        F.lit("R3").alias("rule_id")
    )
    .groupBy("source_file", "rule_id")
    .count()
    .withColumnRenamed("count", "rows_failed")
)

# ------------------------------------------------------------
# 8. R4
# ------------------------------------------------------------
r4_failures = (
    r4_rejected
    .select(
        normalize_rejected_path("_source_file").alias("source_file"),
        F.lit("R4").alias("rule_id")
    )
    .groupBy("source_file", "rule_id")
    .count()
    .withColumnRenamed("count", "rows_failed")
)

# ------------------------------------------------------------
# 9. R5
# ------------------------------------------------------------
r5_failures = (
    r5_rejected
    .select(
        normalize_rejected_path("_source_file").alias("source_file"),
        F.lit("R5").alias("rule_id")
    )
    .groupBy("source_file", "rule_id")
    .count()
    .withColumnRenamed("count", "rows_failed")
)

# ------------------------------------------------------------
# 10. Combine R1-R5
# ------------------------------------------------------------
row_failures = (
    r1_failures
    .unionByName(r2_failures)
    .unionByName(r3_failures)
    .unionByName(r4_failures)
    .unionByName(r5_failures)
)

# ------------------------------------------------------------
# 11. R6 file-level failure
# ------------------------------------------------------------
r6_failures = (
    r6_rejected
    .select(
        F.regexp_replace(
            F.col("source_file"),
            "^dbfs:",
            ""
        ).alias("source_file"),
        F.lit("R6").alias("rule_id"),
        F.lit(1).alias("rows_failed")
    )
)

# ------------------------------------------------------------
# 12. R7 file-level failure
# ------------------------------------------------------------
r7_failures = (
    r7_rejected
    .select(
        F.regexp_replace(
            F.col("source_file"),
            "^dbfs:",
            ""
        ).alias("source_file"),
        F.lit("R7").alias("rule_id"),
        F.lit(1).alias("rows_failed")
    )
)

# ------------------------------------------------------------
# 13. Combine all failures
# ------------------------------------------------------------
all_failures = (
    row_failures
    .unionByName(r6_failures)
    .unionByName(r7_failures)
)

# ------------------------------------------------------------
# 14. Join failures to all load/rule combinations
# ------------------------------------------------------------
gold_orders = (
    gold_orders
    .join(
        all_failures,
        on=["source_file", "rule_id"],
        how="left"
    )
    .withColumn(
        "rows_failed",
        F.coalesce(
            F.col("rows_failed"),
            F.lit(0)
        )
    )
)

# ------------------------------------------------------------
# 15. Rows checked
# ------------------------------------------------------------
gold_orders = gold_orders.withColumn(
    "rows_checked",
    F.when(
        F.col("rule_scope") == "row",
        F.col("rows_in_file")
    ).otherwise(
        F.lit(1)
    )
)

# ------------------------------------------------------------
# 16. Rule status
# ------------------------------------------------------------
gold_orders = gold_orders.withColumn(
    "rule_status",
    F.when(
        F.col("rows_failed") > 0,
        "FAIL"
    ).otherwise(
        "PASS"
    )
)

# ------------------------------------------------------------
# 17. Failure rate
# ------------------------------------------------------------
gold_orders = gold_orders.withColumn(
    "fail_rate",
    F.when(
        F.col("rows_checked") > 0,
        F.col("rows_failed") / F.col("rows_checked")
    ).otherwise(
        F.lit(None).cast("double")
    )
)

# ------------------------------------------------------------
# 18. Final columns
# ------------------------------------------------------------
gold_orders = gold_orders.select(
    F.lit("orders").alias("table_name"),
    "load_id",
    "rule_id",
    "rule_name",
    "rule_scope",
    "rows_checked",
    "rows_failed",
    "fail_rate",
    "rule_status"
)

# ------------------------------------------------------------
# 19. Check row count
# ------------------------------------------------------------
print("Gold order rows:", gold_orders.count())

# ------------------------------------------------------------
# 20. Check failure totals
# ------------------------------------------------------------
display(
    gold_orders
    .groupBy("rule_id", "rule_name")
    .agg(
        F.sum("rows_failed").alias("total_failed"),
        F.sum(
            F.when(
                F.col("rows_failed") > 0,
                1
            ).otherwise(0)
        ).alias("loads_fired")
    )
    .orderBy("rule_id")
)

# ------------------------------------------------------------
# 21. Display full Gold Orders
# ------------------------------------------------------------
display(
    gold_orders
    .orderBy("load_id", "rule_id")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Gold order rows: 112


rule_id,rule_name,total_failed,loads_fired
R1,null_key,0,0
R2,duplicate_order_id,150,1
R3,quantity_not_numeric,40,1
R4,unknown_customer,220,1
R5,order_ts_out_of_range,90,1
R6,column_contract,1,1
R7,row_count_vs_median,1,1


table_name,load_id,rule_id,rule_name,rule_scope,rows_checked,rows_failed,fail_rate,rule_status
orders,20261001052845951_day_01,R1,null_key,row,5000,0,0.0,PASS
orders,20261001052845951_day_01,R2,duplicate_order_id,row,5000,0,0.0,PASS
orders,20261001052845951_day_01,R3,quantity_not_numeric,row,5000,0,0.0,PASS
orders,20261001052845951_day_01,R4,unknown_customer,row,5000,0,0.0,PASS
orders,20261001052845951_day_01,R5,order_ts_out_of_range,row,5000,0,0.0,PASS
orders,20261001052845951_day_01,R6,column_contract,file,1,0,0.0,PASS
orders,20261001052845951_day_01,R7,row_count_vs_median,file,1,0,0.0,PASS
orders,20261001052845951_day_02,R1,null_key,row,5000,0,0.0,PASS
orders,20261001052845951_day_02,R2,duplicate_order_id,row,5000,0,0.0,PASS
orders,20261001052845951_day_02,R3,quantity_not_numeric,row,5000,0,0.0,PASS


In [0]:
gold_orders.filter(
    F.col("rule_id").isin("R2", "R3", "R4", "R5")
).select(
    "load_id",
    "rule_id",
    "rows_checked",
    "rows_failed",
    "rule_status"
).orderBy(
    "rule_id",
    "load_id"
).display()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


load_id,rule_id,rows_checked,rows_failed,rule_status
20261001052845951_day_01,R2,5000,0,PASS
20261001052845951_day_02,R2,5000,0,PASS
20261001052845951_day_03,R2,5000,0,PASS
20261001052845951_day_04,R2,5000,0,PASS
20261001052845951_day_05,R2,5000,0,PASS
20261001052845951_day_06,R2,5000,0,PASS
20261001052845951_day_07,R2,5000,0,PASS
20261001052845951_day_08,R2,5000,0,PASS
20261001052845951_day_09,R2,5000,0,PASS
20261001052845951_day_10,R2,5000,0,PASS


In [0]:
display(
    gold_orders
    .groupBy("rule_id", "rule_name")
    .agg(
        F.sum("rows_failed").alias("total_failed"),
        F.sum(
            F.when(
                F.col("rows_failed") > 0,
                1
            ).otherwise(0)
        ).alias("loads_fired")
    )
    .orderBy("rule_id")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


rule_id,rule_name,total_failed,loads_fired
R1,null_key,0,0
R2,duplicate_order_id,150,1
R3,quantity_not_numeric,40,1
R4,unknown_customer,220,1
R5,order_ts_out_of_range,90,1
R6,column_contract,1,1
R7,row_count_vs_median,1,1


In [0]:
print("R2:", r2_rejected.count())
print("R3:", r3_rejected.count())
print("R4:", r4_rejected.count())
print("R5:", r5_rejected.count())

R2: 150
R3: 40
R4: 220
R5: 90


In [0]:
print("Gold order rows:", gold_orders.count())

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Gold order rows: 112


In [0]:
display(
    gold_orders
    .groupBy("rule_id", "rule_name")
    .agg(
        F.sum("rows_failed").alias("total_failed"),
        F.sum(
            F.when(F.col("rows_failed") > 0, 1).otherwise(0)
        ).alias("loads_fired")
    )
    .orderBy("rule_id")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


rule_id,rule_name,total_failed,loads_fired
R1,null_key,0,0
R2,duplicate_order_id,150,1
R3,quantity_not_numeric,40,1
R4,unknown_customer,220,1
R5,order_ts_out_of_range,90,1
R6,column_contract,1,1
R7,row_count_vs_median,1,1


In [0]:
from pyspark.sql import functions as F

spark.table("gold_dq_results").filter("table_name = 'orders'").groupBy(
    "rule_id",
    "rule_name"
).agg(
    F.sum("rows_failed").alias("total_failed")
).orderBy("rule_id").display()

rule_id,rule_name,total_failed
R1,null_key,0
R2,duplicate_order_id,150
R3,quantity_not_numeric,40
R4,unknown_customer,220
R5,order_ts_out_of_range,90
R6,column_contract,1
R7,row_count_vs_median,1


In [0]:
customer_loads = (
    customers_manifest
    .select(
        "load_id",
        "source_file",
        "rows_in_file"
    )
    .distinct()
)

customer_rules_df = spark.createDataFrame(
    customer_rules,
    ["rule_id", "rule_name", "rule_scope"]
)

# Calculate Customer R7 from the actual customer data
customer_data = spark.table("bronze_customers")

r7_failed = (
    customer_data
    .filter(
        F.col("customer_id").isNull() |
        (F.col("customer_id") == "")
    )
    .count()
)

gold_customers = (
    customer_loads
    .crossJoin(customer_rules_df)
    .withColumn(
        "rows_checked",
        F.when(
            F.col("rule_scope") == "row",
            F.col("rows_in_file")
        ).otherwise(F.lit(1))
    )
    .withColumn(
        "rows_failed",
        F.when(
            F.col("rule_id") == "R7",
            F.lit(r7_failed)
        ).otherwise(F.lit(0))
    )
    .withColumn(
        "fail_rate",
        F.when(
            F.col("rows_checked") > 0,
            F.col("rows_failed") / F.col("rows_checked")
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "rule_status",
        F.when(
            F.col("rows_failed") > 0,
            F.lit("FAIL")
        ).otherwise(F.lit("PASS"))
    )
    .select(
        F.lit("customers").alias("table_name"),
        "load_id",
        "rule_id",
        "rule_name",
        "rule_scope",
        "rows_checked",
        "rows_failed",
        "fail_rate",
        "rule_status"
    )
)

print("Gold customer rows:", gold_customers.count())

display(
    gold_customers.orderBy("load_id", "rule_id")
)

Gold customer rows: 20


table_name,load_id,rule_id,rule_name,rule_scope,rows_checked,rows_failed,fail_rate,rule_status
customers,20261001052845951_cust_day_01,R1,null_key,row,300,0,0.0,PASS
customers,20261001052845951_cust_day_01,R7,row_count_vs_median,file,1,0,0.0,PASS
customers,20261001052845951_cust_day_02,R1,null_key,row,300,0,0.0,PASS
customers,20261001052845951_cust_day_02,R7,row_count_vs_median,file,1,0,0.0,PASS
customers,20261001052845951_cust_day_03,R1,null_key,row,300,0,0.0,PASS
customers,20261001052845951_cust_day_03,R7,row_count_vs_median,file,1,0,0.0,PASS
customers,20261001052845951_cust_day_04,R1,null_key,row,300,0,0.0,PASS
customers,20261001052845951_cust_day_04,R7,row_count_vs_median,file,1,0,0.0,PASS
customers,20261001052845951_cust_day_05,R1,null_key,row,300,0,0.0,PASS
customers,20261001052845951_cust_day_05,R7,row_count_vs_median,file,1,0,0.0,PASS


We are ready to build gold_dq_results

In [0]:
gold_dq_results = (
    gold_orders
    .unionByName(gold_customers)
)

print("Gold DQ rows:", gold_dq_results.count())

display(
    gold_dq_results
    .orderBy("table_name", "load_id", "rule_id")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Gold DQ rows: 132


table_name,load_id,rule_id,rule_name,rule_scope,rows_checked,rows_failed,fail_rate,rule_status
customers,20261001052845951_cust_day_01,R1,null_key,row,300,0,0.0,PASS
customers,20261001052845951_cust_day_01,R7,row_count_vs_median,file,1,0,0.0,PASS
customers,20261001052845951_cust_day_02,R1,null_key,row,300,0,0.0,PASS
customers,20261001052845951_cust_day_02,R7,row_count_vs_median,file,1,0,0.0,PASS
customers,20261001052845951_cust_day_03,R1,null_key,row,300,0,0.0,PASS
customers,20261001052845951_cust_day_03,R7,row_count_vs_median,file,1,0,0.0,PASS
customers,20261001052845951_cust_day_04,R1,null_key,row,300,0,0.0,PASS
customers,20261001052845951_cust_day_04,R7,row_count_vs_median,file,1,0,0.0,PASS
customers,20261001052845951_cust_day_05,R1,null_key,row,300,0,0.0,PASS
customers,20261001052845951_cust_day_05,R7,row_count_vs_median,file,1,0,0.0,PASS


In [0]:
# Verify Gold DQ row count

gold_dq_count = gold_dq_results.count()

print("Gold DQ rows:", gold_dq_count)

assert gold_dq_count == 132

print("PASS: gold_dq_results contains exactly 132 rows.")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Gold DQ rows: 132
PASS: gold_dq_results contains exactly 132 rows.


In [0]:
# Export the final Gold DQ results as a single CSV
export_path = "/Volumes/workspace/dbsf_2570002/raw_files/gold_dq_results_export_final"

gold_dq_results.coalesce(1).write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(export_path)

print("CSV exported to:", export_path)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


CSV exported to: /Volumes/workspace/dbsf_2570002/raw_files/gold_dq_results_export_final


In [0]:
csv_check = spark.read \
    .option("header", "true") \
    .csv(export_path)

display(
    csv_check
    .filter(
        (F.col("table_name") == "orders") &
        (F.col("rule_id").isin("R2", "R3", "R4", "R5"))
    )
    .groupBy("rule_id", "rule_name")
    .agg(
        F.sum(F.col("rows_failed").cast("long")).alias("total_failed")
    )
    .orderBy("rule_id")
)

rule_id,rule_name,total_failed
R2,duplicate_order_id,150
R3,quantity_not_numeric,40
R4,unknown_customer,220
R5,order_ts_out_of_range,90


In [0]:
from pyspark.sql import functions as F

spark.table("gold_dq_results").groupBy(
    "table_name",
    "rule_id",
    "rule_name"
).agg(
    F.sum("rows_failed").alias("total_failed")
).orderBy(
    "table_name",
    "rule_id"
).display()

table_name,rule_id,rule_name,total_failed
customers,R1,null_key,0
customers,R7,row_count_vs_median,0
orders,R1,null_key,0
orders,R2,duplicate_order_id,150
orders,R3,quantity_not_numeric,40
orders,R4,unknown_customer,220
orders,R5,order_ts_out_of_range,90
orders,R6,column_contract,1
orders,R7,row_count_vs_median,1


In [0]:
# Verify total row-scope order failures

row_scope_order_failures = (
    gold_dq_results
    .filter(
        (F.col("table_name") == "orders") &
        (F.col("rule_scope") == "row")
    )
    .agg(
        F.sum("rows_failed").alias("total_rows_failed")
    )
    .first()["total_rows_failed"]
)

print(
    "Total row-scope order failures:",
    row_scope_order_failures
)

assert row_scope_order_failures == 500

print(
    "PASS: Row-scope order failures total exactly 500."
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Total row-scope order failures: 500
PASS: Row-scope order failures total exactly 500.


In [0]:
gold_dq_results.write.mode("overwrite").saveAsTable(
    "gold_dq_results"
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
gold_dq_results = (
    gold_orders
    .unionByName(gold_customers)
)

print("Gold DQ rows:", gold_dq_results.count())

display(
    gold_dq_results
    .orderBy("table_name", "load_id", "rule_id")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Gold DQ rows: 132


table_name,load_id,rule_id,rule_name,rule_scope,rows_checked,rows_failed,fail_rate,rule_status
customers,20261001052845951_cust_day_01,R1,null_key,row,300,0,0.0,PASS
customers,20261001052845951_cust_day_01,R7,row_count_vs_median,file,1,0,0.0,PASS
customers,20261001052845951_cust_day_02,R1,null_key,row,300,0,0.0,PASS
customers,20261001052845951_cust_day_02,R7,row_count_vs_median,file,1,0,0.0,PASS
customers,20261001052845951_cust_day_03,R1,null_key,row,300,0,0.0,PASS
customers,20261001052845951_cust_day_03,R7,row_count_vs_median,file,1,0,0.0,PASS
customers,20261001052845951_cust_day_04,R1,null_key,row,300,0,0.0,PASS
customers,20261001052845951_cust_day_04,R7,row_count_vs_median,file,1,0,0.0,PASS
customers,20261001052845951_cust_day_05,R1,null_key,row,300,0,0.0,PASS
customers,20261001052845951_cust_day_05,R7,row_count_vs_median,file,1,0,0.0,PASS


In [0]:
from pyspark.sql import functions as F

spark.table("gold_dq_results").groupBy(
    "table_name",
    "rule_id",
    "rule_name"
).agg(
    F.sum("rows_failed").alias("total_failed")
).orderBy(
    "table_name",
    "rule_id"
).display()

table_name,rule_id,rule_name,total_failed
customers,R1,null_key,0
customers,R7,row_count_vs_median,0
orders,R1,null_key,0
orders,R2,duplicate_order_id,150
orders,R3,quantity_not_numeric,40
orders,R4,unknown_customer,220
orders,R5,order_ts_out_of_range,90
orders,R6,column_contract,1
orders,R7,row_count_vs_median,1


In [0]:
print(
    "Saved Gold DQ rows:",
    spark.table("gold_dq_results").count()
)

Saved Gold DQ rows: 132


In [0]:
print("Gold DQ rows:", spark.table("gold_dq_results").count())

Gold DQ rows: 132


In [0]:
display(
    spark.table("gold_dq_results")
    .groupBy("table_name")
    .count()
    .orderBy("table_name")
)

table_name,count
customers,20
orders,112


In [0]:
display(
    spark.table("gold_dq_results")
    .groupBy("rule_id", "rule_name")
    .agg(
        F.sum("rows_failed").alias("total_failed")
    )
    .orderBy("rule_id")
)

rule_id,rule_name,total_failed
R1,null_key,0
R2,duplicate_order_id,150
R3,quantity_not_numeric,40
R4,unknown_customer,220
R5,order_ts_out_of_range,90
R6,column_contract,1
R7,row_count_vs_median,1


gold_dq_results table is done. Hence, we'll start building gold_freshness table

Gold_Freshness Table
--------------------

In [0]:
from pyspark.sql import functions as F

# Reconstruct silver_orders from persisted tables
silver_orders = spark.table("silver_orders")

# ============================================================
# GOLD FRESHNESS
# ============================================================

# ------------------------------------------------------------
# 1. Find the newest order date
#    This is our reference/check timestamp.
# ------------------------------------------------------------
reference_ts = (
    silver_orders
    .select(
        F.max(
            F.expr("try_to_timestamp(order_ts, 'yyyy-MM-dd')")
        ).alias("reference_ts")
    )
    .collect()[0]["reference_ts"]
)

print("Freshness reference timestamp:", reference_ts)


# ------------------------------------------------------------
# 2. Latest business timestamp for orders
# ------------------------------------------------------------
latest_orders = (
    silver_orders
    .select(
        F.max(
            F.expr("try_to_timestamp(order_ts, 'yyyy-MM-dd')")
        ).alias("last_business_ts")
    )
    .withColumn(
        "table_name",
        F.lit("orders")
    )
)


# ------------------------------------------------------------
# 3. Latest business timestamp for customers
# ------------------------------------------------------------
latest_customers = (
    spark.table("bronze_customers")
    .select(
        F.max(
            F.to_timestamp("signup_date", "yyyy-MM-dd")
        ).alias("last_business_ts")
    )
    .withColumn(
        "table_name",
        F.lit("customers")
    )
)


# ------------------------------------------------------------
# 4. Build freshness table
# ------------------------------------------------------------
gold_freshness = (
    latest_orders
    .unionByName(latest_customers)
    .withColumn(
        "check_ts",
        F.lit(reference_ts).cast("timestamp")
    )
    .withColumn(
        "hours_behind",
        (
            F.unix_timestamp("check_ts")
            - F.unix_timestamp("last_business_ts")
        ) / 3600
    )
    .withColumn(
        "expected_interval_hours",
        F.lit(24.0)
    )
    .withColumn(
        "freshness_status",
F.when(
    F.col("hours_behind") <= F.col("expected_interval_hours"),
    "FRESH"
)
.otherwise("STALE")
    )
    .select(
        "table_name",
        "check_ts",
        "last_business_ts",
        "hours_behind",
        "expected_interval_hours",
        "freshness_status"
    )
)


# ------------------------------------------------------------
# 5. Display result
# ------------------------------------------------------------
display(
    gold_freshness
    .orderBy("table_name")
)


# ------------------------------------------------------------
# 6. Save Gold freshness table
# ------------------------------------------------------------
gold_freshness.write \
    .mode("overwrite") \
    .saveAsTable("gold_freshness")

Freshness reference timestamp: 2025-06-16 00:00:00


table_name,check_ts,last_business_ts,hours_behind,expected_interval_hours,freshness_status
customers,2025-06-16T00:00:00.000Z,2025-06-10T00:00:00.000Z,144.0,24.0,STALE
orders,2025-06-16T00:00:00.000Z,2025-06-16T00:00:00.000Z,0.0,24.0,FRESH


In [0]:
display(
    spark.table("gold_freshness")
)

table_name,check_ts,last_business_ts,hours_behind,expected_interval_hours,freshness_status
customers,2025-06-16T00:00:00.000Z,2025-06-10T00:00:00.000Z,144.0,24.0,STALE
orders,2025-06-16T00:00:00.000Z,2025-06-16T00:00:00.000Z,0.0,24.0,FRESH


Our Gold layer is finally complete. However, as per the instructions, our gold layer must answer three questions - 
1. Rows rejected per load, per rule, per day
2. How many hours behind the newest data is each table, and is that too long?
3. Which rule fires most - and is the rule wrong rather than the data?

Question 1
-----

In [0]:
from pyspark.sql import functions as F

gold_rejections_by_day = (
    spark.table("silver_order_rejections")
    .withColumn(
        "day",
        F.regexp_extract(
            F.col("_source_file"),
            r"/(day_\d+)/",
            1
        )
    )
    .groupBy("day", "rule_id", "_load_id", "reason")
    .agg(
        F.count("*").alias("rows_rejected")
    )
    .orderBy("day", "rule_id")
)

display(gold_rejections_by_day)

day,rule_id,_load_id,reason,rows_rejected
day_12,R3,20261001052845951,quantity_not_numeric,40
day_13,R2,20261001052845951,duplicate_order_id,150
day_16,R4,20261001052845951,unknown_customer,220
day_16,R5,20261001052845951,order_ts_out_of_range,90


Question 2 has already been answered where we found out - 
-------------------------------------------------------

| table_name | hours_behind | expected_interval_hours | freshness_status |
| ---------- | -----------: | ----------------------: | ---------------- |
| orders     |          0.0 |                      24 | FRESH            |
| customers  |        144.0 |                      24 | STALE            |


Question 3
----------

In [0]:
display(
    spark.table("silver_order_rejections")
    .filter(F.col("rule_id") == "R4")
    .groupBy("customer_id")
    .count()
    .orderBy(F.desc("count"))
)

customer_id,count
C99999,220


So the answer is R4 (Unknown Customer) fires the most, with 220 rejected rows. All 220 rows contain customer ID C99999, which is not present in the customer master. Therefore, the rule is correctly identifying invalid customer references rather than incorrectly rejecting valid data.

Now that we are done building bronze, silver and gold layer, we'll now move on to migrating to SnowFlake for further processing. For that, we need to take Gold layer outputs from here to SnowFlake

In [0]:
gold_dq_results = spark.table("gold_dq_results")

gold_dq_results.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("/Volumes/workspace/dbsf_2570002/raw_files/gold_dq_results_export")

In [0]:
display(
    dbutils.fs.ls("/Volumes/workspace/dbsf_2570002/raw_files/gold_dq_results_export")
)

path,name,size,modificationTime
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/gold_dq_results_export/_committed_1537123923509646990,_committed_1537123923509646990,204,1790816132000
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/gold_dq_results_export/_committed_1635296840269630083,_committed_1635296840269630083,204,1790807467000
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/gold_dq_results_export/_committed_1670705868956274002,_committed_1670705868956274002,205,1790794716000
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/gold_dq_results_export/_committed_1942886020708588631,_committed_1942886020708588631,203,1790734042000
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/gold_dq_results_export/_committed_292178009159065586,_committed_292178009159065586,204,1790815207000
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/gold_dq_results_export/_committed_2934373806408701778,_committed_2934373806408701778,203,1790760384000
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/gold_dq_results_export/_committed_3115278004231054058,_committed_3115278004231054058,204,1790834337000
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/gold_dq_results_export/_committed_3800804927632100212,_committed_3800804927632100212,204,1790789263000
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/gold_dq_results_export/_committed_4635638402513042781,_committed_4635638402513042781,205,1790816695000
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/gold_dq_results_export/_committed_5206415643723860886,_committed_5206415643723860886,204,1790802875000


In [0]:
gold_freshness.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("/Volumes/workspace/dbsf_2570002/raw_files/gold_freshness_export")

In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/workspace/dbsf_2570002/raw_files/gold_freshness_export"
    )
)

path,name,size,modificationTime
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/gold_freshness_export/_SUCCESS,_SUCCESS,0,1790834361000
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/gold_freshness_export/_committed_8537469860739193616,_committed_8537469860739193616,114,1790834360000
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/gold_freshness_export/_started_8537469860739193616,_started_8537469860739193616,0,1790834360000
dbfs:/Volumes/workspace/dbsf_2570002/raw_files/gold_freshness_export/part-00000-tid-8537469860739193616-172a4a67-e6e6-4c83-ae6c-a331ecf3157b-4163-1-c000.csv,part-00000-tid-8537469860739193616-172a4a67-e6e6-4c83-ae6c-a331ecf3157b-4163-1-c000.csv,240,1790834360000
